In [ ]:
# Part 1: Imports
# ipynb

# 1. Standard library imports
import argparse
import math
import os
from pprint import pprint
import random
import shutil
import time

# 2. Third-party library imports
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 3. PyTorch core and utilities
import torch
from torch import autograd
from torch.distributions.multivariate_normal import MultivariateNormal
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
!pip install tensorboard
from torch.utils.tensorboard import SummaryWriter

# 4. Torchvision imports
from torchvision import datasets, transforms
from torchvision.utils import save_image

# 5. Device configuration
cuda = torch.cuda.is_available()
device = torch.device("cuda" if cuda else "cpu")


In [ ]:
import numpy as np
import os
import shutil
import torch
from torch.nn import functional as F
from torchvision import datasets, transforms
from torch.utils import data
import torch.utils.data as Data
from torch.distributions.multivariate_normal import MultivariateNormal
from PIL import Image
import math

# device = torch.device("cuda:5" if(torch.cuda.is_available()) else "cpu")
bce = torch.nn.BCEWithLogitsLoss(reduction='none')
bce3 =  torch.nn.BCELoss(reduction='none')

def mask_threshold(x):
  x = (x+0.5).int().float()
  return x
  
def label_cov(labels):
	cov = torch.from_numpy(np.cov(labels, rowvar = False)).to(device)
	return cov
 
def get_labelcov_prior(batchsize, cov):
  #print(cov)
  v = torch.zeros(batchsize, cov.size()[0], cov.size()[1])
  for i in range(batchsize):
    v[i] = cov
  mean = torch.zeros(batchsize, cov.size()[1])
  return mean, v
 
def vector_expand(v):
	V = torch.zeros(v.size()[0],v.size()[1],v.size()[1]).to(device)
	for i in range(v.size()[0]):
		for j in range(v.size()[1]):
			V[i,j,j] = v[i,j]
	return V
 
def block_matmul(a, b):
	return None
  
def multivariate_sample(m,cov):
  m = m.reshape(m.size()[0],4)
  z = torch.zeros(m.size())
  for i in range(z.size()[0]):
    z[i] = MultivariateNormal(m[i].cpu(), cov[i].cpu()).sample()
  return z.to(device)
  
def kl_multinormal_cov(qm,qv, pm, pv):
	KL = torch.zeros(qm.size()[0]).to(device)
	for i in range(qm.size()[0]):
		#print(torch.det(qv[i].cpu()))
		KL[i] = 0.5 * (torch.log(torch.det(pv[i])) - torch.log(torch.det(qv[i])) +
		torch.trace(torch.inverse(pv[i]))*torch.trace(torch.inverse(qv[i])) + 
		torch.norm(qm[i])*torch.norm(pv[i], p=1))
	return KL
 
 
def conditional_sample_gaussian(m,v):
	sample = torch.randn(m.size()).to(m.device)
	z = m + (v**0.5)*sample
	return z



def gumbel_sample(m, v, temp=0.5):
	# Sample from Gumbel
	u = torch.rand_like(m)
	g = - torch.log(- torch.log(u + v) + v)

	# Gumbel-Softmax sample
	z = F.softmax((m + g) / temp, dim=-1)
	z = z.view(-1, 32 * 4)

	return z


def gaussian_log_prob(samples, mean, var):
	""" Returns the log probability of a specified Gaussian for a tensor of samples """
	return -torch.log(var) - 0.5 * np.log(2*np.pi) - 0.5 * ((samples - mean))**2 / var


def condition_gaussian_parameters(h, dim = 1):
	#print(h.size())
	m, h = torch.split(h, h.size(1) // 2, dim=1)
	m = torch.reshape(m, [-1, 3, 4])
	h = torch.reshape(h, [-1, 3, 4])
	v = F.softplus(h) + 1e-8
	return m, v

def condition_prior(scale, label, dim):
	mean = torch.ones(label.size()[0],label.size()[1], dim)
	var = torch.ones(label.size()[0],label.size()[1], dim)
	for i in range(label.size()[0]):
		for j in range(label.size()[1]):
			mul = (float(label[i][j])-scale[j][0])/(scale[j][1]-0)
			mean[i][j] = torch.ones(dim)*mul
			#mean[i][j] = torch.ones(dim)*label[i][j].detach().cpu()
			# if j==2 or j == 3:
			# 	mean[i][j] = torch.randn(dim)
			var[i][j] = torch.ones(dim)*1
	return mean, var


def causal_prior(scale, label, dim, A, mask=None):
	mean = torch.ones(label.size()[0], label.size()[1], dim)
	var = torch.ones(label.size()[0], label.size()[1], dim)
	for i in range(label.size()[0]):
		I = torch.eye(4).to(device)
		inp = A.to(device).t() + I

		# num_parents = torch.count_nonzero(inp, dim=1).reshape((dim, 1))
		num_parents = torch.tensor([1, 1, 3, 3]).reshape((dim, 1)).to(device)

		norm_label = (label[i].to(device) - torch.tensor(scale[:, 0]).to(device)) / (
			torch.tensor(scale[:, 1]).to(device))
		out = torch.matmul(inp.float(), norm_label.float().to(device)).reshape((dim, 1))

		fin = torch.div(out, num_parents).repeat(1, dim)
		# fin = out

		mean[i] = fin
		var[i] = torch.ones(dim, dim)

	# if i == 0:
	#     print(label[i])
	#     print(mean[i])
	#     exit(0)

	return mean, var


def compute_kl(z_1, z_2, logvar_1, logvar_2):
	var_1 = torch.exp(logvar_1)
	var_2 = torch.exp(logvar_2)
	return var_1/var_2 + torch.square(z_2-z_1)/var_2 - 1 + logvar_2 - logvar_1


def scm_prior(scale, label, dim, A, mask=None):
	mean = torch.ones(label.size()[0], label.size()[1], dim)
	var = torch.ones(label.size()[0], label.size()[1], dim)
	for i in range(label.size()[0]):
		I = torch.eye(4).to(device)
		inp = A.to(device).t() + I

		num_parents = torch.count_nonzero(inp, dim=1).reshape((dim, 1))
		# num_parents = torch.tensor([1, 1, 3, 3]).reshape((dim, 1)).to(device)

		norm_label = (label[i].to(device) - torch.tensor(scale[:, 0]).to(device)) / (
			torch.tensor(scale[:, 1]).to(device))
		out = torch.matmul(inp.float(), norm_label.float().to(device)).reshape((dim, 1))

		fin = torch.div(out, num_parents).repeat(1, dim)
		# fin = out
        
		mean[i] = fin
		var[i] = torch.ones(dim, dim)

	# if i == 0:
	#     print(label[i])
	#     print(mean[i])
	#     exit(0)

	return mean, var




# Conditional Structure Prior
def structural_condition_prior(scale, label, dim, A): # CHANGE TO WHEN WE ARE INTERVENING ON 3RD OR 4TH CONCEPT TO HARDCODE IT
    mean = torch.ones(label.size()[0], label.size()[1], dim)
    var = torch.ones(label.size()[0], label.size()[1], dim)
    I = torch.eye(4).to(label.device)
    for i in range(label.size()[0]):
        for j in range(label.size()[1]):

            inp = A.to(label.device).t() + I

            num_parents = torch.count_nonzero((inp), dim=1)
            norm_label = (label[i].to(label.device) - torch.tensor(scale[:, 0]).to(label.device)) / (torch.tensor(scale[:, 1]).to(label.device))

            mul = torch.matmul((inp).float(), norm_label.float().to(label.device))[j]
            mul = mul / num_parents[j] # averaging

            mean[i][j] = torch.ones(dim) * mul.item()
            
            if j == 3:
                mean[i][j] = torch.randn(dim)

            var[i][j] = torch.ones(dim) * 1
    return mean, var


 
def bce2(r, x):
	return x * torch.log(r + 1e-7) + (1 - x) * torch.log(1 - r + 1e-7)

################################################################################
# Please familiarize yourself with the code below.
#
# Note that the notation is
# argument: argument_type: argument_shape
#
# Furthermore, the expected argument_shape is only a guideline. You're free to
# pass in inputs that violate the expected argument_shape provided you know
# what you're doing
################################################################################

def sample_multivariate(cov, loc = None):
	# if loc == None:
	# 	loc = torch.zeros((cov.shape[0], cov.shape[0]))
	latent_code = torch.distributions.multivariate_normal.MultivariateNormal(loc, covariance_matrix=cov, precision_matrix=None, scale_tril=None, validate_args=None)
	return latent_code

def get_covariance_matrix(A):
	# requirements: A must be torcj
	assert A.size()[1] == A.size()[2]
	I = torch.zeros(A.size()).to(device)
	i = torch.eye(n = A.size()[1]).to(device)
	for j in range(A.size()[0]):
		I[j] = torch.inverse(torch.mm(torch.t((A[j]-i)), (A[j]-i)))
	
	return I


def sample_gaussian(m, v):
	"""
	Element-wise application reparameterization trick to sample from Gaussian

	Args:
		m: tensor: (batch, ...): Mean
		v: tensor: (batch, ...): Variance

	Return:
		z: tensor: (batch, ...): Samples
	"""
	################################################################################
	# TODO: Modify/complete the code here
	# Sample z
	################################################################################

	################################################################################
	# End of code modification
	################################################################################
	sample = torch.randn(m.shape).to(device)
	

	z = m + (v**0.5)*sample
	return z

def log_normal(x, m, var):
	"""
	Computes the elem-wise log probability of a Gaussian and then sum over the
	last dim. Basically we're assuming all dims are batch dims except for the
	last dim.

	Args:
		x: tensor: (batch, ..., dim): Observation
		m: tensor: (batch, ..., dim): Mean
		v: tensor: (batch, ..., dim): Variance

	Return:
		kl: tensor: (batch1, batch2, ...): log probability of each sample. Note
			that the summation dimension (dim=-1) is not kept
	"""

	const = -0.5*x.size(-1)*torch.log(2*torch.tensor(np.pi))
	#print(const.size())
	log_det = -0.5*torch.sum(var, dim = -1)
	# print(log_det)
	# print(f'Variance: {m}')
	log_exp = -0.5*torch.sum( (x - m)**2/torch.exp(var), dim = -1)
	# print(f'Exp: {(x - m)**2}')
	log_prob = const + log_det + log_exp

	return log_prob



# def gaussian_log_prob(x, m, logvar):
# 	"""
# 	Computes the elem-wise log probability of a Gaussian and then sum over the
# 	last dim. Basically we're assuming all dims are batch dims except for the
# 	last dim.

# 	Args:
# 		x: tensor: (batch, ..., dim): Observation
# 		m: tensor: (batch, ..., dim): Mean
# 		v: tensor: (batch, ..., dim): Variance

# 	Return:
# 		kl: tensor: (batch1, batch2, ...): log probability of each sample. Note
# 			that the summation dimension (dim=-1) is not kept
# 	"""
# 	#print("q_m", m.size())
# 	#print("q_v", v.size())
# 	const = -0.5*x.size(-1)*torch.log(2*torch.tensor(np.pi))
# 	#print(const.size())
# 	log_det = -0.5*torch.sum(logvar, dim = -1)
# 	#print("log_det", log_det.size())
# 	log_exp = -0.5*torch.sum( (x - m)**2/torch.exp(logvar), dim = -1)

# 	log_prob = const + log_det + log_exp

# 	return log_prob



def log_gaussian(x, mu, log_var):
	"""
	Returns the log pdf of a normal distribution parametrised
	by mu and log_var evaluated at x. (Univariate distribution)
	:param x: point to evaluate
	:param mu: mean of distribution
	:param log_var: log variance of distribution
	:return: log N(x|µ,σ)
	"""
	log_pdf = - 0.5 * np.log(2 * np.pi) - (log_var + 1e-8) / 2 - ((x - mu)**2 + 1e-8) / (2 * torch.exp(log_var))
	# print('Size log_pdf:', log_pdf.shape)
	return torch.sum(log_pdf, dim=-1)



def log_normal_mixture(z, m, v):
	"""
	Computes log probability of a uniformly-weighted Gaussian mixture.

	Args:
		z: tensor: (batch, dim): Observations
		m: tensor: (batch, mix, dim): Mixture means
		v: tensor: (batch, mix, dim): Mixture variances

	Return:
		log_prob: tensor: (batch,): log probability of each sample
	"""
	################################################################################
	# TODO: Modify/complete the code here
	# Compute the uniformly-weighted mixture of Gaussians density for each sample
	# in the batch
	################################################################################
	z = z.unsqueeze(1)
	log_probs = log_normal(z, m, v)
	#print("log_probs_mix", log_probs.shape)

	log_prob = log_mean_exp(log_probs, 1)
	#print("log_prob_mix", log_prob.size())

	################################################################################
	# End of code modification
	################################################################################
	return log_prob


def gaussian_parameters(h, dim=-1):
	"""
	Converts generic real-valued representations into mean and variance
	parameters of a Gaussian distribution

	Args:
		h: tensor: (batch, ..., dim, ...): Arbitrary tensor
		dim: int: (): Dimension along which to split the tensor for mean and
			variance

	Returns:z
		m: tensor: (batch, ..., dim / 2, ...): Mean
		v: tensor: (batch, ..., dim / 2, ...): Variance
	"""
	m, h = torch.split(h, h.size(dim) // 2, dim=dim)
	v = F.softplus(h) + 1e-8
	return m, v


def log_bernoulli_with_logits(x, logits):
	"""
	Computes the log probability of a Bernoulli given its logits

	Args:
		x: tensor: (batch, dim): Observation
		logits: tensor: (batch, dim): Bernoulli logits

	Return:
		log_prob: tensor: (batch,): log probability of each sample
	"""
	log_prob = -bce(input=logits, target=x).sum(-1)
	return log_prob


def cross_entropy(x, logits):
	"""
	Computes the log probability of a Bernoulli given its logits

	Args:
		x: tensor: (batch, dim): Observation
		logits: tensor: (batch, dim): Bernoulli logits

	Return:
		log_prob: tensor: (batch,): log probability of each sample
	"""
	log_prob = -bce(input=logits, target=x).sum(-1)
	return log_prob


def log_bernoulli_with_logits_nosigmoid(x, logits):
	"""
	Computes the log probability of a Bernoulli given its logits

	Args:
		x: tensor: (batch, dim): Observation
		logits: tensor: (batch, dim): Bernoulli logits

	Return:
		log_prob: tensor: (batch,): log probability of each sample
	"""

	log_prob = bce2(logits, x).sum(-1)

	return log_prob




def kl_cat(q, log_q, log_p):
	"""
	Computes the KL divergence between two categorical distributions

	Args:
		q: tensor: (batch, dim): Categorical distribution parameters
		log_q: tensor: (batch, dim): Log of q
		log_p: tensor: (batch, dim): Log of p

	Return:
		kl: tensor: (batch,) kl between each sample
	"""
	element_wise = (q * (log_q - log_p))
	kl = element_wise.sum(-1)
	return kl


def kl_normal(qm, qv, pm, pv):
	"""
	Computes the elem-wise KL divergence between two normal distributions KL(q || p) and
	sum over the last dimension

	Args:
		qm: tensor: (batch, dim): q mean
		qv: tensor: (batch, dim): q variance
		pm: tensor: (batch, dim): p mean
		pv: tensor: (batch, dim): p variance

	Return:
		kl: tensor: (batch,): kl between each sample
	"""
	element_wise = 0.5 * (torch.log(pv) - torch.log(qv) + qv / pv + (qm - pm).pow(2) / pv - 1)
	kl = element_wise.sum(-1)
	#print("log var1", qv)
	return kl


def log_prob(qm, qv, pm, pv):
	"""
	Computes the elem-wise KL divergence between two normal distributions KL(q || p) and
	sum over the last dimension

	Args:
		qm: tensor: (batch, dim): q mean
		qv: tensor: (batch, dim): q variance
		pm: tensor: (batch, dim): p mean
		pv: tensor: (batch, dim): p variance

	Return:
		kl: tensor: (batch,): kl between each sample
	"""
	element_wise = 0.5 * (pv - qv + torch.exp(qv) / torch.exp(pv) + (qm - pm).pow(2) / torch.exp(pv) - 1)
	kl = element_wise.sum(-1)
	#print("log var1", qv)
	return kl



def duplicate(x, rep):
	"""
	Duplicates x along dim=0

	Args:
		x: tensor: (batch, ...): Arbitrary tensor
		rep: int: (): Number of replicates. Setting rep=1 returns orignal x
  z 
	Returns:
		_: tensor: (batch * rep, ...): Arbitrary replicated tensor
	"""
	return x.expand(rep, *x.shape).reshape(-1, *x.shape[1:])


def log_mean_exp(x, dim):
	"""
	Compute the log(mean(exp(x), dim)) in a numerically stable manner

	Args:
		x: tensor: (...): Arbitrary tensor
		dim: int: (): Dimension along which mean is computed

	Return:
		_: tensor: (...): log(mean(exp(x), dim))
	"""
	return log_sum_exp(x, dim) - np.log(x.size(dim))


def log_sum_exp(x, dim=0):
	"""
	Compute the log(sum(exp(x), dim)) in a numerically stable manner

	Args:
		x: tensor: (...): Arbitrary tensor
		dim: int: (): Dimension along which sum is computed

	Return:
		_: tensor: (...): log(sum(exp(x), dim))
	"""
	max_x = torch.max(x, dim)[0]
	new_x = x - max_x.unsqueeze(dim).expand_as(x)
	return max_x + (new_x.exp().sum(dim)).log()


def load_model_by_name(model, global_step):
	"""
	Load a model based on its name model.name and the checkpoint iteration step

	Args:
		model: Model: (): A model
		global_step: int: (): Checkpoint iteration
	"""
	file_path = os.path.join('checkpoints',
							 model.name,
							 'model-{:05d}.pt'.format(global_step))
	print(file_path)
	state = torch.load(file_path, map_location='cpu')
	# print(state)
	model.load_state_dict(state)
	print("Loaded from {}".format(file_path))


################################################################################
# No need to read/understand code beyond this point. Unless you want to.
# But do you tho ¯\_(�?_/¯
################################################################################


def evaluate_lower_bound(model, labeled_test_subset, run_iwae=True):
	check_model = isinstance(model, VAE) or isinstance(model, GMVAE) or isinstance(model, LVAE)
	assert check_model, "This function is only intended for VAE and GMVAE"

	print('*' * 80)
	print("LOG-LIKELIHOOD LOWER BOUNDS ON TEST SUBSET")
	print('*' * 80)

	xl, _ = labeled_test_subset
	torch.manual_seed(0)
	xl = torch.bernoulli(xl)

	def detach_torch_tuple(args):
		return (v.detach() for v in args)

	def compute_metrics(fn, repeat):
		metrics = [0, 0, 0]
		for _ in range(repeat):
			niwae, kl, rec = detach_torch_tuple(fn(xl))
			metrics[0] += niwae / repeat
			metrics[1] += kl / repeat
			metrics[2] += rec / repeat
		return metrics

	# Run multiple times to get low-var estimate
	nelbo, kl, rec = compute_metrics(model.negative_elbo_bound, 100)
	print("NELBO: {}. KL: {}. Rec: {}".format(nelbo, kl, rec))

	if run_iwae:
		for iw in [1, 10, 100, 1000]:
			repeat = max(100 // iw, 1) # Do at least 100 iterations
			fn = lambda x: model.negative_iwae_bound(x, iw)
			niwae, kl, rec = compute_metrics(fn, repeat)
			print("Negative IWAE-{}: {}".format(iw, niwae))


def evaluate_classifier(model, test_set):
	check_model = isinstance(model, SSVAE)
	assert check_model, "This function is only intended for SSVAE"

	print('*' * 80)
	print("CLASSIFICATION EVALUATION ON ENTIRE TEST SET")
	print('*' * 80)

	X, y = test_set
	pred = model.cls.classify(X)
	accuracy = (pred.argmax(1) == y).float().mean()
	print("Test set classification accuracy: {}".format(accuracy))


def save_model_by_name(model, global_step):
	save_dir = os.path.join('checkpoints', model.name)
	if not os.path.exists(save_dir):
		os.makedirs(save_dir)
	file_path = os.path.join(save_dir, 'model-{:05d}.pt'.format(global_step))
	state = model.state_dict()
	torch.save(state, file_path)
	print('Saved to {}'.format(file_path))


def prepare_writer(model_name, overwrite_existing=False):
	log_dir = os.path.join('logs', model_name)
	save_dir = os.path.join('checkpoints', model_name)
	if overwrite_existing:
		delete_existing(log_dir)
		delete_existing(save_dir)
	# Sadly, I've been told *not* to use tensorflow :<
	# writer = tf.summary.FileWriter(log_dir)
	writer = None
	return writer


def log_summaries(writer, summaries, global_step):
	pass # Sad :<
	# for tag in summaries:
	#	 val = summaries[tag]
	#	 tf_summary = tf.Summary.Value(tag=tag, simple_value=val)
	#	 writer.add_summary(tf.Summary(value=[tf_summary]), global_step)
	# writer.flush()


def delete_existing(path):
	if os.path.exists(path):
		print("Deleting existing path: {}".format(path))
		shutil.rmtree(path)


def reset_weights(m):
	try:
		m.reset_parameters()
	except AttributeError:
		pass


def get_mnist_data(device, use_test_subset=True):
	preprocess = transforms.ToTensor()
	train_loader = torch.utils.data.DataLoader(
		datasets.MNIST('data', train=True, download=True, transform=preprocess),
		batch_size=100,
		shuffle=True)
	test_loader = torch.utils.data.DataLoader(
		datasets.MNIST('data', train=False, download=True, transform=preprocess),
		batch_size=100,
		shuffle=True)

	# Create pre-processed training and test sets
	X_train = train_loader.dataset.train_data.to(device).reshape(-1, 784).float() / 255
	y_train = train_loader.dataset.train_labels.to(device)
	X_test = test_loader.dataset.test_data.to(device).reshape(-1, 784).float() / 255
	y_test = test_loader.dataset.test_labels.to(device)

	# Create supervised subset (deterministically chosen)
	# This subset will serve dual purpose of log-likelihood evaluation and
	# semi-supervised learning. Pretty hacky. Don't judge :<
	X = X_test if use_test_subset else X_train
	y = y_test if use_test_subset else y_train

	xl, yl = [], []
	for i in range(10):
		idx = y == i
		idx_choice = get_mnist_index(i, test=use_test_subset)
		xl += [X[idx][idx_choice]]
		yl += [y[idx][idx_choice]]
	xl = torch.cat(xl).to(device)
	yl = torch.cat(yl).to(device)
	yl = yl.new(np.eye(10)[yl])
	labeled_subset = (xl, yl)

	return train_loader, labeled_subset, (X_test, y_test)


def get_mnist_index(i, test=True):
	# Obviously *hand*-coded
	train_idx = np.array([[2732,2607,1653,3264,4931,4859,5827,1033,4373,5874],
						  [5924,3468,6458,705,2599,2135,2222,2897,1701,537],
						  [2893,2163,5072,4851,2046,1871,2496,99,2008,755],
						  [797,659,3219,423,3337,2745,4735,544,714,2292],
						  [151,2723,3531,2930,1207,802,2176,2176,1956,3622],
						  [3560,756,4369,4484,1641,3114,4984,4353,4071,4009],
						  [2105,3942,3191,430,4187,2446,2659,1589,2956,2681],
						  [4180,2251,4420,4870,1071,4735,6132,5251,5068,1204],
						  [3918,1167,1684,3299,2767,2957,4469,560,5425,1605],
						  [5795,1472,3678,256,3762,5412,1954,816,2435,1634]])

	test_idx = np.array([[684,559,629,192,835,763,707,359,9,723],
						 [277,599,1094,600,314,705,551,87,174,849],
						 [537,845,72,777,115,976,755,448,850,99],
						 [984,177,755,797,659,147,910,423,288,961],
						 [265,697,639,544,543,714,244,151,675,510],
						 [459,882,183,28,802,128,128,53,550,488],
						 [756,273,335,388,617,42,442,543,888,257],
						 [57,291,779,430,91,398,611,908,633,84],
						 [203,324,774,964,47,639,131,972,868,180],
						 [1000,846,143,660,227,954,791,719,909,373]])

	if test:
		return test_idx[i]

	else:
		return train_idx[i]


def get_svhn_data(device):
	preprocess = transforms.ToTensor()
	train_loader = torch.utils.data.DataLoader(
		datasets.SVHN('data', split='extra', download=True, transform=preprocess),
		batch_size=100,
		shuffle=True)

	return train_loader, (None, None), (None, None)


def gumbel_softmax(logits, tau, eps=1e-8):
	U = torch.rand_like(logits)
	gumbel = -torch.log(-torch.log(U + eps) + eps)
	y = logits + gumbel
	y = F.softmax(y / tau, dim=1)
	return y

class DeterministicWarmup(object):
	"""
	Linear deterministic warm-up as described in
	[Sønderby 2016].
	"""
	def __init__(self, n=100, t_max=1):
		self.t = 0
		self.t_max = t_max
		self.inc = 1/n

	def __iter__(self):
		return self

	def __next__(self):
		t = self.t + self.inc

		self.t = self.t_max if t > self.t_max else t
		return self.t

class FixedSeed:
	def __init__(self, seed):
		self.seed = seed
		self.state = None

	def __enter__(self):
		self.state = np.random.get_state()
		np.random.seed(self.seed)

	def __exit__(self, exc_type, exc_value, traceback):
		np.random.set_state(self.state)


In [ ]:
import numpy as np
import scipy
from sklearn import ensemble
from sklearn import metrics
from sklearn import linear_model
import torch
from munkres import Munkres
from scipy.optimize import linear_sum_assignment


import numpy as np
import scipy as sp
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

import numpy as np




def generate_batch_factor_code(ground_truth_data, representation_function, num_points, random_state, batch_size):
    """Sample a single training sample based on a mini-batch of ground-truth data.

    Args:
    ground_truth_data: GroundTruthData to be sampled from.
    representation_function: Function that takes observation as input and
      outputs a representation.
    num_points: Number of points to sample.
    random_state: Numpy random state used for randomness.
    batch_size: Batchsize to sample points.

    Returns:
    representations: Codes (num_codes, num_points)-np array.
    factors: Factors generating the codes (num_factors, num_points)-np array.
    """
    representations = None
    factors = None
    i = 0
    while i < num_points:
        num_points_iter = min(num_points - i, batch_size)
        current_factors, current_observations = \
            ground_truth_data.sample(num_points_iter, random_state)
        if i == 0:
            factors = current_factors
            representations = representation_function(current_observations)
        else:
            factors = np.vstack((factors, current_factors))
            representations = np.vstack((representations,
                                       representation_function(
                                           current_observations)))
        i += num_points_iter
    return np.transpose(representations), np.transpose(factors)


# def make_discretizer(target, num_bins=gin.REQUIRED, discretizer_fn=gin.REQUIRED):

#     return discretizer_fn(target, num_bins)



def compute_irs(rep, y, diff_quantile=0.99):
    """Computes the Interventional Robustness Score.

    Args:
    ground_truth_data: GroundTruthData to be sampled from.
    representation_function: Function that takes observations as input and
      outputs a dim_representation sized representation for each observation.
    random_state: Numpy random state used for randomness.
    artifact_dir: Optional path to directory where artifacts can be saved.
    diff_quantile: Float value between 0 and 1 to decide what quantile of diffs
      to select (use 1.0 for the version in the paper).
    num_train: Number of points used for training.
    batch_size: Batch size for sampling.

    Returns:
    Dict with IRS and number of active dimensions.
    """

    # mus, ys = generate_batch_factor_code(ground_truth_data,
    #                                          representation_function, num_train,
    #                                          random_state, batch_size)
    # assert mus.shape[1] == num_train


    if not rep.any():
        irs_score = 0.0
    else:
        irs_score = scalable_disentanglement_score(y.T, rep.T, diff_quantile)["avg_score"]

    score_dict = {}
    score_dict["IRS"] = irs_score
    score_dict["num_active_dims"] = np.sum(rep)
    
    return score_dict


def _drop_constant_dims(ys):
    """Returns a view of the matrix `ys` with dropped constant rows."""
    ys = np.asarray(ys)
    if ys.ndim != 2:
        raise ValueError("Expecting a matrix.")

    variances = ys.var(axis=1)
    active_mask = variances > 0.
    
    return ys[active_mask, :]


def scalable_disentanglement_score(gen_factors, latents, diff_quantile=0.99):
    """Computes IRS scores of a dataset.

    Assumes no noise in X and crossed generative factors (i.e. one sample per
    combination of gen_factors). Assumes each g_i is an equally probable
    realization of g_i and all g_i are independent.

    Args:
    gen_factors: Numpy array of shape (num samples, num generative factors),
      matrix of ground truth generative factors.
    latents: Numpy array of shape (num samples, num latent dimensions), matrix
      of latent variables.
    diff_quantile: Float value between 0 and 1 to decide what quantile of diffs
      to select (use 1.0 for the version in the paper).

    Returns:
    Dictionary with IRS scores.
    """
    num_gen = gen_factors.shape[1]
    num_lat = latents.shape[1]

    # Compute normalizer.
    max_deviations = np.max(np.abs(latents - latents.mean(axis=0)), axis=0)
    cum_deviations = np.zeros([num_lat, num_gen])
    for i in range(num_gen):
        unique_factors = np.unique(gen_factors[:, i], axis=0)
        assert unique_factors.ndim == 1
        num_distinct_factors = unique_factors.shape[0]
        for k in range(num_distinct_factors):
            # Compute E[Z | g_i].
            match = gen_factors[:, i] == unique_factors[k]
            e_loc = np.mean(latents[match, :], axis=0)

            # Difference of each value within that group of constant g_i to its mean.
            diffs = np.abs(latents[match, :] - e_loc)
            max_diffs = np.percentile(diffs, q=diff_quantile*100, axis=0)
            cum_deviations[:, i] += max_diffs
        cum_deviations[:, i] /= num_distinct_factors
    # Normalize value of each latent dimension with its maximal deviation.
    normalized_deviations = cum_deviations / max_deviations[:, np.newaxis]
    irs_matrix = 1.0 - normalized_deviations
    disentanglement_scores = irs_matrix.max(axis=1)
    if np.sum(max_deviations) > 0.0:
        avg_score = np.average(disentanglement_scores, weights=max_deviations)
    else:
        avg_score = np.mean(disentanglement_scores)

    parents = irs_matrix.argmax(axis=1)
    score_dict = {}
    score_dict["disentanglement_scores"] = disentanglement_scores
    score_dict["avg_score"] = avg_score
    score_dict["parents"] = parents
    score_dict["IRS_matrix"] = irs_matrix
    score_dict["max_deviations"] = max_deviations
    
    return score_dict


def _compute_dci(mus_train, ys_train, mus_test, ys_test):
    """Computes score based on both training and testing codes and factors."""
    scores = {}
    importance_matrix, train_err, test_err = compute_importance_gbt(
        mus_train, ys_train, mus_test, ys_test)
    assert importance_matrix.shape[0] == mus_train.shape[0]
    assert importance_matrix.shape[1] == ys_train.shape[0]
    scores["informativeness_train"] = train_err
    scores["informativeness_test"] = test_err
    disent, code_importance = disentanglement(importance_matrix)
    scores["disentanglement"] = disent
    scores["completeness"] = completeness(importance_matrix)
    return scores, importance_matrix, code_importance


def compute_importance_gbt(x_train, y_train, x_test, y_test):
    """Compute importance based on gradient boosted trees."""
    num_factors = y_train.shape[0]
    num_codes = x_train.shape[0]
    importance_matrix = np.zeros(shape=[num_codes, num_factors],
                                 dtype=np.float64)
    train_loss = []
    test_loss = []
    for i in range(num_factors):
        # from xgboost import XGBClassifier
        # model = XGBClassifier()
        # model = ensemble.GradientBoostingClassifier()
        model = ensemble.GradientBoostingRegressor()
        model.fit(x_train.T, y_train[i, :])
        importance_matrix[:, i] = np.abs(model.feature_importances_)
        train_loss.append(np.mean(model.predict(x_train.T) == y_train[i, :]))
        test_loss.append(np.mean(model.predict(x_test.T) == y_test[i, :]))
    return importance_matrix, np.mean(train_loss), np.mean(test_loss)


def disentanglement_per_code(importance_matrix):
  """Compute disentanglement score of each code."""
  # importance_matrix is of shape [num_codes, num_factors].
  return 1. - scipy.stats.entropy(importance_matrix.T + 1e-11,
                                  base=importance_matrix.shape[1])


def disentanglement(importance_matrix):
  """Compute the disentanglement score of the representation."""
  per_code = disentanglement_per_code(importance_matrix)
  if importance_matrix.sum() == 0.:
    importance_matrix = np.ones_like(importance_matrix)
  code_importance = importance_matrix.sum(axis=1) / importance_matrix.sum()
    
  return np.sum(per_code*code_importance), code_importance


def completeness_per_factor(importance_matrix):
  """Compute completeness of each factor."""
  # importance_matrix is of shape [num_codes, num_factors].
  return 1. - scipy.stats.entropy(importance_matrix + 1e-11,
                                  base=importance_matrix.shape[0])


def completeness(importance_matrix):
  """"Compute completeness of the representation."""
  per_factor = completeness_per_factor(importance_matrix)
  if importance_matrix.sum() == 0.:
    importance_matrix = np.ones_like(importance_matrix)
  factor_importance = importance_matrix.sum(axis=0) / importance_matrix.sum()
  return np.sum(per_factor*factor_importance)


def MCC(Z, Zp):
    n = np.shape(Z)[1]
    #     print (n)
    rho_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            rho_matrix[i, j] = np.abs(np.corrcoef(Z[:, i], Zp[:, j])[0, 1])

    r, c = linear_sum_assignment(-rho_matrix)

    return np.mean(rho_matrix[r, c])


def r2_disentanglement(z, hz, mode = "r2", reorder=None):
    """Measure how well hz reconstructs z measured either by the Coefficient of Determination or the
    Pearson/Spearman correlation coefficient."""

    assert mode in ("r2", "adjusted_r2", "pearson", "spearman")

    if mode == "r2":
        # print(z[0].shape)
        # print(hz[0].shape)
        # exit(0)
        r2_i = []
        for i in range(z.shape[0]):
            r2_i.append(metrics.r2_score(z[i], hz[i]))
            print(metrics.r2_score(z[i], hz[i]))

        return sum(r2_i) / len(r2_i)
    elif mode == "adjusted_r2":
        r2 = metrics.r2_score(z, hz)
        # number of data samples
        n = z.shape[0]
        # number of predictors, i.e. features
        p = z.shape[1]
        adjusted_r2 = 1.0 - (1.0 - r2) * (n - 1) / (n - p - 1)
        return adjusted_r2, None
    elif mode in ("spearman", "pearson"):
        dim = z.shape[-1]

        if mode == "spearman":
            raw_corr, pvalue = scipy.stats.spearmanr(z, hz)
        else:
            raw_corr = np.corrcoef(z.T, hz.T)
        corr = raw_corr[:dim, dim:]

        if reorder:
            # effectively computes MCC
            munk = Munkres()
            indexes = munk.compute(-np.absolute(corr))

            sort_idx = np.zeros(dim)
            hz_sort = np.zeros(z.shape)
            for i in range(dim):
                sort_idx[i] = indexes[i][1]
                hz_sort[:, i] = hz[:, indexes[i][1]]

            if mode == "spearman":
                raw_corr, pvalue = scipy.stats.spearmanr(z, hz_sort)
            else:
                raw_corr = np.corrcoef(z.T, hz_sort.T)

            corr = raw_corr[:dim, dim:]

        return np.diag(np.abs(corr)).mean(), corr

    
    
def linear_disentanglement(z, hz, mode="r2", train_test_split=None):
    """Calculate disentanglement up to linear transformations.
    Args:
        z: Ground-truth latents.
        hz: Reconstructed latents.
        mode: Can be r2, pearson, spearman
        train_test_split: Use first half to train linear model, second half to test.
            Is only relevant if there are less samples then latent dimensions.
    """

    if torch.is_tensor(hz):
        hz = hz.detach().cpu().numpy()
    if torch.is_tensor(z):
        z = z.detach().cpu().numpy()

    # assert isinstance(z, np.ndarray), "Either pass a torch tensor or numpy array as z"
    # assert isinstance(hz, np.ndarray), "Either pass a torch tensor or numpy array as hz"

    # split z, hz to get train and test set for linear model
    if train_test_split:
        n_train = len(z) // 2
        z_1 = z[:n_train]
        hz_1 = hz[:n_train]
        z_2 = z[n_train:]
        hz_2 = hz[n_train:]
    else:
        z_1 = z
        hz_1 = hz
        z_2 = z
        hz_2 = hz

    model = linear_model.LinearRegression()
    model.fit(hz_1, z_1)

    hz_2 = model.predict(hz_2)

    inner_result = _disentanglement(z_2, hz_2, mode=mode, reorder=False)

    return inner_result, (z_2, hz_2)


def _disentanglement(z, hz, mode="r2", reorder=None):
    """Measure how well hz reconstructs z measured either by the Coefficient of Determination or the
    Pearson/Spearman correlation coefficient."""

    # assert mode in ("r2", "adjusted_r2", "pearson", "spearman")

    if mode == "r2":
        return metrics.r2_score(z, hz), None
    elif mode == "adjusted_r2":
        r2 = metrics.r2_score(z, hz)
        # number of data samples
        n = z.shape[0]
        # number of predictors, i.e. features
        p = z.shape[1]
        adjusted_r2 = 1.0 - (1.0 - r2) * (n - 1) / (n - p - 1)
        return adjusted_r2, None
    elif mode in ("spearman", "pearson"):
        dim = z.shape[-1]

        if mode == "spearman":
            raw_corr, pvalue = sp.stats.spearmanr(z, hz)
        else:
            raw_corr = np.corrcoef(z.T, hz.T)
        corr = raw_corr[:dim, dim:]

        if reorder:
            # effectively computes MCC
            munk = Munkres()
            indexes = munk.compute(-np.absolute(corr))

            sort_idx = np.zeros(dim)
            hz_sort = np.zeros(z.shape)
            for i in range(dim):
                sort_idx[i] = indexes[i][1]
                hz_sort[:, i] = hz[:, indexes[i][1]]

            if mode == "spearman":
                raw_corr, pvalue = sp.stats.spearmanr(z, hz_sort)
            else:
                raw_corr = np.corrcoef(z.T, hz_sort.T)

            corr = raw_corr[:dim, dim:]

        return np.diag(np.abs(corr)).mean(), corr
  
    
def permutation_disentanglement(
    z,
    hz,
    mode="r2",
    rescaling=True,
    solver="naive",
    sign_flips=True,
    cache_permutations=None,
):
    """Measure disentanglement up to permutations by either using the Munkres solver
    or naively trying out every possible permutation.
    Args:
        z: Ground-truth latents.
        hz: Reconstructed latents.
        mode: Can be r2, pearson, spearman
        rescaling: Rescale every individual latent to maximize the agreement
            with the ground-truth.
        solver: How to find best possible permutation. Either use Munkres algorithm
            or naively test every possible permutation.
        sign_flips: Only relevant for `naive` solver. Also include sign-flips in
            set of possible permutations to test.
        cache_permutations: Only relevant for `naive` solver. Cache permutation matrices
            to allow faster access if called multiple times.
    """

    assert solver in ("naive", "munkres")
    if mode == "r2" or mode == "adjusted_r2":
        assert solver == "naive", "R2 coefficient is only supported with naive solver"

    if cache_permutations and not hasattr(
        permutation_disentanglement, "permutation_matrices"
    ):
        permutation_disentanglement.permutation_matrices = dict()

    if torch.is_tensor(hz):
        hz = hz.detach().cpu().numpy()
    if torch.is_tensor(z):
        z = z.detach().cpu().numpy()

    assert isinstance(z, np.ndarray), "Either pass a torch tensor or numpy array as z"
    assert isinstance(hz, np.ndarray), "Either pass a torch tensor or numpy array as hz"

    def test_transformation(T, reorder):
        # measure the r2 score for one transformation

        Thz = hz @ T
        if rescaling:
            assert z.shape == hz.shape
            # find beta_j that solve Y_ij = X_ij beta_j
            Y = z
            X = hz

            beta = np.diag((Y * X).sum(0) / (X ** 2).sum(0))

            Thz = X @ beta

        return _disentanglement(z, Thz, mode=mode, reorder=reorder), Thz

    def gen_permutations(n):
        # generate all possible permutations w/ or w/o sign flips

        def gen_permutation_single_row(basis, row, sign_flips=False):
            # generate all possible permutations w/ or w/o sign flips for one row
            # assuming the previous rows are already fixed
            basis = basis.clone()
            basis[row] = 0
            for i in range(basis.shape[-1]):
                # skip possible columns if there is already an entry in one of
                # the previous rows
                if torch.sum(torch.abs(basis[:row, i])) > 0:
                    continue
                signs = [1]
                if sign_flips:
                    signs += [-1]

                for sign in signs:
                    T = basis.clone()
                    T[row, i] = sign

                    yield T

        def gen_permutations_all_rows(basis, current_row=0, sign_flips=False):
            # get all possible permutations for all rows

            for T in gen_permutation_single_row(basis, current_row, sign_flips):
                if current_row == len(basis) - 1:
                    yield T.numpy()
                else:
                    # generate all possible permutations of all other rows
                    yield from gen_permutations_all_rows(T, current_row + 1, sign_flips)

        basis = torch.zeros((n, n))

        yield from gen_permutations_all_rows(basis, sign_flips=sign_flips)

    n = z.shape[-1]
    # use cache to speed up repeated calls to the function
    if cache_permutations and not solver == "munkres":
        key = (rescaling, n)
        if not key in permutation_disentanglement.permutation_matrices:
            permutation_disentanglement.permutation_matrices[key] = list(
                gen_permutations(n)
            )
        permutations = permutation_disentanglement.permutation_matrices[key]
    else:
        if solver == "naive":
            permutations = list(gen_permutations(n))
        elif solver == "munkres":
            permutations = [np.eye(n, dtype=z.dtype)]

    scores = []

    # go through all possible permutations and check r2 score
    for T in permutations:
        scores.append(test_transformation(T, solver == "munkres"))

    return max(scores, key=lambda x: x[0][0])

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import MultivariateNormal, Normal, Uniform
import torch.nn.functional as F
# device = torch.device("cuda:5" if(torch.cuda.is_available()) else "cpu")
from codebase import utils as ut


class MLP(nn.Module):
    """ a simple 4-layer MLP """

    def __init__(self, nin, nout, nh):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nin, nh),
            nn.ReLU(),
            nn.Linear(nh, nh),
            nn.ReLU(),
            nn.Linear(nh, nout),
            nn.Sigmoid(),
        )
    def forward(self, x, mask):
        return self.net(x * mask)
    
    
# FOR DIFFEOMORPHIC SCM-VAE
class MultivariateCausalFlow(nn.Module):
    def __init__(self, dim, k, C=None, net_class=MLP, nh=100, scale=True, shift=True):
        super().__init__()
        self.dim = dim
        self.k = k

        self.C = C
        self.A = (torch.eye(self.C.shape[0]) - self.C)
        
        if scale:
            self.s_cond = net_class(self.dim*self.k, self.k, 100)
        if shift: 
            self.t_cond = net_class(self.dim*self.k, self.k, 100)
        
        self.z_int_prior = Normal(0.0, 1.0)

    
    def forward(self, e, target=None, value=None):

        total_dims = e.shape[1]*e.shape[2]
        log_det = torch.zeros(e.size(0)).to(e.device)
        p_logprob = torch.zeros(e.size(0)).to(e.device)
        batch_size = e.shape[0]
        z = torch.zeros(batch_size, self.dim, self.k).to(e.device)
        
        
        for i in range(self.dim):    
            if 1 in self.C[:, i]: # does it have any parents (z_3)
                # mask = self.C[:, i].reshape(self.dim).to(device) # [1, 1, 0, 0]
                mask = self.C[:, i].repeat(self.k, 1).T.reshape(total_dims).to(e.device)
            elif 1 not in self.C[:, i] or target == i: # doesnt have parents
                mask = torch.zeros(total_dims).to(e.device)
            
            # compute slope and offset
            s = self.s_cond(z.reshape(-1, total_dims), mask).reshape(batch_size, self.k) # slope
            t = self.t_cond(z.reshape(-1, total_dims), mask).reshape(batch_size, self.k) # offset

            # slope and offset transformation (affine transformation)
            z[:, i, :] = torch.exp(s) * e[:, i, :].reshape(batch_size, self.k) + t
            if target is not None and value is not None:
                # temp = z.reshape(batch_size, self.dim*self.k)
                # temp[:, 77] = 0.1
                # z = temp.reshape(batch_size, self.dim, self.k)
                # temp = z.clone()
                # temp[:, 2, 19] = value[:, 19]
                # z = temp.clone()
                z[:, target, :] = value
                #z[:, 0, :] = value
            log_det += torch.sum(s, dim=1) # dz / de
            
        return z, log_det
    
    def backward(self, z, target=None, value=None):
        
        total_dims = z.shape[1]*z.shape[2]
        log_det = torch.zeros(z.size(0)).to(z.device)
        p_logprob = torch.zeros(z.size(0)).to(z.device)
        batch_size = z.shape[0]
        e = torch.zeros(batch_size, self.dim, self.k).to(z.device)
        
        
        for i in range(self.dim):
            
            if 1 in self.C[:, i]: # does it have any parents (z_3)
                # mask = self.C[:, i].reshape(self.dim).to(device) # [1, 1, 0, 0]
                mask = self.C[:, i].repeat(self.k, 1).T.reshape(total_dims).to(e.device)
            elif 1 not in self.C[:, i] or target == i: # doesnt have parents
                mask = torch.zeros(total_dims).to(e.device)
            
            # compute slope and offset
            s = self.s_cond(z.reshape(-1, total_dims), mask).reshape(batch_size, self.k) # slope
            t = self.t_cond(z.reshape(-1, total_dims), mask).reshape(batch_size, self.k) # offset
            
        
            # slope and offset transformation (affine transformation)
            e[:, i, :] = torch.exp(-s) * (z[:, i, :].reshape(batch_size, self.k) - t)
            # if target is not None and value is not None:
            #     z[:, target, :] = torch.ones(1, self.k).to(e.device) * value
                # z[:, target, :] = value.to(device)
            log_det -= torch.mean(s, dim=1) # dz / de
            
        return z, log_det
    
    
    
    def forward_interv(self, e, I):
        total_dims = e.shape[1]*e.shape[2]
        log_det = torch.zeros(e.size(0)).to(e.device)
        p_logprob = torch.zeros(e.size(0)).to(e.device)
        batch_size = e.shape[0]
        z = torch.zeros(batch_size, self.dim, self.dim).to(e.device)
        
        
        for i in range(self.dim):
            
            interv_mask = (I[:, i] == 1).to(e.device) #[T, F, F, F]
            # print(interv_mask)
            
            if 1 in self.C[:, i]: # does it have any parents (z_3)
                # mask = self.C[:, i].reshape(self.dim).to(device) # [1, 1, 0, 0]
                mask = self.C[:, i].repeat(4, 1).T.reshape(total_dims).to(e.device)
            else: # doesnt have parents
                mask = torch.zeros(total_dims).to(e.device)
            
            # Standard Gaussian sampled intervention
            #z_base = torch.randn(e[:, i].shape).to(device)
            
            # z_base = torch.randn(e[:, i, :].shape).to(device)
            
            # if z_base_inf is not None:
            #     z_base = z_base_inf

            # Intervention
            # e[:, i] = torch.where(interv_mask.reshape(batch_size), z_base.clone(), e[:, i].clone())
            # z[:, i, :] = torch.where(interv_mask.reshape(batch_size, 4), z_base.clone(), z[:, i, :].clone())
            z[:, i, :] = torch.ones(1, 4).to(e.device) * 3
  
            s = torch.where(interv_mask.reshape(batch_size, 4), 
                            self.s_cond(z.reshape(-1, total_dims), torch.zeros(total_dims).to(e.device)).reshape(batch_size, self.dim), 
                            self.s_cond(z.reshape(-1, total_dims), mask).reshape(batch_size, self.dim))
            
            t = torch.where(interv_mask.reshape(batch_size, 4), 
                            self.t_cond(z.reshape(-1, total_dims), torch.zeros(total_dims).to(e.device)).reshape(batch_size, self.dim), 
                            self.t_cond(z.reshape(-1, total_dims), mask).reshape(batch_size, self.dim))
        
            
            # slope and offset transformation (affine transformation)
            z[:, i, :] = torch.exp(s) * (e[:, i, :] - t)
            

        return z
    

    
# FOR DIFFEOMORPHIC SCM-VAE
class PriorMultivariateCausalFlow(nn.Module):
    def __init__(self, dim, k, C=None, net_class=MLP, nh=100, scale=True, shift=True):
        super().__init__()
        self.dim = dim
        self.k = k

        self.C = C
        self.A = (torch.eye(self.C.shape[0]) - self.C)
        
        if scale:
            self.s_cond = net_class(self.dim*self.k, self.k, 100)
        if shift: 
            self.t_cond = net_class(self.dim*self.k, self.k, 100)
        
        self.z_int_prior = Normal(0.0, 1.0)

    
    def forward(self, e, latent=None, target=None, value=None):

        total_dims = e.shape[1]*e.shape[2]
        log_det = torch.zeros(e.size(0)).to(e.device)
        p_logprob = torch.zeros(e.size(0)).to(e.device)
        batch_size = e.shape[0]
        z = torch.zeros(batch_size, self.dim, self.k).to(e.device)
        
        
        for i in range(self.dim):    
            if 1 in self.C[:, i]: # does it have any parents (z_3)
                # mask = self.C[:, i].reshape(self.dim).to(device) # [1, 1, 0, 0]
                mask = self.C[:, i].repeat(self.k, 1).T.reshape(total_dims).to(e.device)
            elif 1 not in self.C[:, i] or target == i: # doesnt have parents
                mask = torch.zeros(total_dims).to(e.device)
            
            # compute slope and offset
            s = self.s_cond(latent.reshape(-1, total_dims), mask).reshape(batch_size, self.k) # slope
            t = self.t_cond(latent.reshape(-1, total_dims), mask).reshape(batch_size, self.k) # offset

            # slope and offset transformation (affine transformation)
            z[:, i, :] = torch.exp(s) * e[:, i, :].reshape(batch_size, self.k) + t
            if target is not None and value is not None:
                # temp = z.reshape(batch_size, self.dim*self.k)
                # temp[:, 77] = 0.1
                # z = temp.reshape(batch_size, self.dim, self.k)
                # temp = z.clone()
                # temp[:, 2, 19] = value[:, 19]
                # z = temp.clone()
                z[:, target, :] = value
                #z[:, 0, :] = value
            log_det += torch.sum(s, dim=1) # dz / de
            
        return z, log_det
    
    def backward(self, z, target=None, value=None):
        
        total_dims = z.shape[1]*z.shape[2]
        log_det = torch.zeros(z.size(0)).to(z.device)
        p_logprob = torch.zeros(z.size(0)).to(z.device)
        batch_size = z.shape[0]
        e = torch.zeros(batch_size, self.dim, self.k).to(z.device)
        
        
        for i in range(self.dim):
            
            if 1 in self.C[:, i]: # does it have any parents (z_3)
                # mask = self.C[:, i].reshape(self.dim).to(device) # [1, 1, 0, 0]
                mask = self.C[:, i].repeat(self.k, 1).T.reshape(total_dims).to(e.device)
            elif 1 not in self.C[:, i] or target == i: # doesnt have parents
                mask = torch.zeros(total_dims).to(e.device)
            
            # compute slope and offset
            s = self.s_cond(z.reshape(-1, total_dims), mask).reshape(batch_size, self.k) # slope
            t = self.t_cond(z.reshape(-1, total_dims), mask).reshape(batch_size, self.k) # offset
            
        
            # slope and offset transformation (affine transformation)
            e[:, i, :] = torch.exp(-s) * (z[:, i, :].reshape(batch_size, self.k) - t)
            # if target is not None and value is not None:
            #     z[:, target, :] = torch.ones(1, self.k).to(e.device) * value
                # z[:, target, :] = value.to(device)
            log_det -= torch.mean(s, dim=1) # dz / de
            
        return z, log_det
    
    
    
    def forward_interv(self, e, I):
        total_dims = e.shape[1]*e.shape[2]
        log_det = torch.zeros(e.size(0)).to(e.device)
        p_logprob = torch.zeros(e.size(0)).to(e.device)
        batch_size = e.shape[0]
        z = torch.zeros(batch_size, self.dim, self.dim).to(e.device)
        
        
        for i in range(self.dim):
            
            interv_mask = (I[:, i] == 1).to(e.device) #[T, F, F, F]
            # print(interv_mask)
            
            if 1 in self.C[:, i]: # does it have any parents (z_3)
                # mask = self.C[:, i].reshape(self.dim).to(device) # [1, 1, 0, 0]
                mask = self.C[:, i].repeat(4, 1).T.reshape(total_dims).to(e.device)
            else: # doesnt have parents
                mask = torch.zeros(total_dims).to(e.device)
            
            # Standard Gaussian sampled intervention
            #z_base = torch.randn(e[:, i].shape).to(device)
            
            # z_base = torch.randn(e[:, i, :].shape).to(device)
            
            # if z_base_inf is not None:
            #     z_base = z_base_inf

            # Intervention
            # e[:, i] = torch.where(interv_mask.reshape(batch_size), z_base.clone(), e[:, i].clone())
            # z[:, i, :] = torch.where(interv_mask.reshape(batch_size, 4), z_base.clone(), z[:, i, :].clone())
            z[:, i, :] = torch.ones(1, 4).to(e.device) * 3
  
            s = torch.where(interv_mask.reshape(batch_size, 4), 
                            self.s_cond(z.reshape(-1, total_dims), torch.zeros(total_dims).to(e.device)).reshape(batch_size, self.dim), 
                            self.s_cond(z.reshape(-1, total_dims), mask).reshape(batch_size, self.dim))
            
            t = torch.where(interv_mask.reshape(batch_size, 4), 
                            self.t_cond(z.reshape(-1, total_dims), torch.zeros(total_dims).to(e.device)).reshape(batch_size, self.dim), 
                            self.t_cond(z.reshape(-1, total_dims), mask).reshape(batch_size, self.dim))
        
            
            # slope and offset transformation (affine transformation)
            z[:, i, :] = torch.exp(s) * (e[:, i, :] - t)
            

        return z

    
    
# FOR ILCM
class CausalAffineAutoregFlow(nn.Module):
    def __init__(self, dim, C, net_class=MLP, nh=100, scale=True, shift=True):
        super().__init__()
        self.dim = dim
        # self.s_cond = lambda x: x.new_zeros(x.size(0), self.dim)
        # self.t_cond = lambda x: x.new_zeros(x.size(0), self.dim)
        self.C = C
        if scale:
            self.s_cond = net_class(self.dim, 1, 100)
        if shift: 
            self.t_cond = net_class(self.dim, 1, 100)
        
        self.z_int_prior = Normal(0.0, 1.0)
        # self.z_int_prior = Uniform(0.0, 1.0)


    
    def forward(self, e):
        log_det = torch.zeros(e.size(0)).to(device)
        p_logprob = torch.zeros(e.size(0)).to(device)
        batch_size = e.shape[0]
        z = torch.zeros(e.shape).to(device)
        
        # set z to e
        # z = e.clone()
        
        for i in range(self.dim):
            
            if 1 in self.C[:, i]: # does it have any parents (z_3)
                mask = self.C[:, i].reshape(self.dim).to(device) # [1, 1, 0, 0]
            else: # doesnt have parents
                mask = torch.zeros(self.dim).to(device)
            
            # compute slope and offset
            s = self.s_cond(z, mask).reshape(z.shape[0]) # slope
            t = self.t_cond(z, mask).reshape(z.shape[0]) # offset
            
            # print(s.shape)
            # print(z[:, i].shape)
            
            # slope and offset transformation (affine transformation)
            z[:, i] = torch.exp(s) * e[:, i] + t # z1 = s * e_1 + t, z_3 = s * e_3 + t
            # print(s)
            # f1(e_1, pai=0) = s*e_1 + t = z1
            # f2 --- z2
            # f3(e_3, pai = (z_1, z_2)) = s * e_3 + t, [z1, z2, 0, 0]
            # f4
            
            log_det += s # dz / de
            
        return z, log_det
    
    
    def backward(self, z, I, z_base_inf=None):
        log_det = torch.zeros(z.size(0)).to(device)
        p_logprob = torch.zeros(z.size(0)).to(device)
        batch_size = z.shape[0]
        e = torch.zeros(z.shape).to(device)
        
        # [e1, e2, e3, e4] = [z1, z2, z3, z4]
        # e = z.clone()
        
        # [z1, z2, e3, e4]
        # [z1, z2, z3, e4]
        # [z1, z2, z3, z4]
        # []
#         z_base = torch.randn(batch_size).to(device)
            
#         if z_base_inf is not None:
#             z_base = z_base_inf
#         z_base = torch.randn(batch_size).to(device)
            
#         if z_base_inf is not None:
#             z_base = z_base_inf
        for i in range(self.dim):
            
            interv_mask = (I[:, i] == 1).unsqueeze(-1).to(device) #[T, F, F, F]
            
            if 1 in self.C[:, i]: # if it has parents
                mask = self.C[:, i].reshape(self.dim).to(device)
            else: # if it doesnt
                mask = torch.zeros(self.dim).to(device)
            
            
            # Standard Gaussian sampled intervention
            #z_base = torch.randn(e[:, i].shape).to(device)
            
            z_base = torch.randn(e[:, i].shape).to(device)
            
            if z_base_inf is not None:
                z_base = z_base_inf

            # Intervention
            # e[:, i] = torch.where(interv_mask.reshape(batch_size), z_base.clone(), e[:, i].clone())
            z[:, i] = torch.where(interv_mask.reshape(batch_size), z_base.clone(), z[:, i].clone())

            # z3 = z3'
            
            # compute slope and offset as a function of e\i
#             s = torch.where(interv_mask.reshape(batch_size), 
#                             self.s_cond(e, torch.zeros(self.dim).to(device)).reshape(z.shape[0]), 
#                             self.s_cond(e, mask).reshape(z.shape[0]))
            
#             t = torch.where(interv_mask.reshape(batch_size), 
#                             self.t_cond(e, torch.zeros(self.dim).to(device)).reshape(z.shape[0]), 
#                             self.t_cond(e, mask).reshape(z.shape[0]))
            
            s = torch.where(interv_mask.reshape(batch_size), 
                            self.s_cond(z, torch.zeros(self.dim).to(device)).reshape(z.shape[0]), 
                            self.s_cond(z, mask).reshape(z.shape[0]))
            
            t = torch.where(interv_mask.reshape(batch_size), 
                            self.t_cond(z, torch.zeros(self.dim).to(device)).reshape(z.shape[0]), 
                            self.t_cond(z, mask).reshape(z.shape[0]))
        
            
            # slope and offset transformation (affine transformation)
            e[:, i] = torch.exp(-s) * (z[:, i] - t)
            
            s_new = torch.where(interv_mask.reshape(batch_size), s.to(device), torch.zeros(s.shape).to(device))
            z_val = torch.where(interv_mask.reshape(batch_size), self.z_int_prior.log_prob(z_base).to(device), torch.zeros(z[:, i].shape).to(device))
            
            # s = self.s_cond(z, mask).reshape(z.shape[0])
            # t = self.t_cond(z, mask).reshape(z.shape[0])
            
            log_det -= s_new
            p_logprob += z_val
            # p_logprob += ut.gaussian_log_prob(z_val, torch.zeros(batch_size).to(device), torch.ones(batch_size).to(device))
            # p_logprob += ut.log_normal(z_val, torch.zeros(batch_size).to(device), torch.ones(batch_size).to(device)) 
            
        return e, p_logprob, log_det

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from codebase import utils as ut
from torch import autograd, nn, optim
from torch import nn
from torch.nn import functional as F
from torch.nn import Linear

import numpy as np
import torch
import torch.nn.functional as F
from codebase import utils as ut
from torch import autograd, nn, optim
from torch import nn
from torch.nn import functional as F
from torch.nn import Linear

# Gaussian Encoder
class Encoder(nn.Module):
    def __init__(self, z_dim, channel=4, y_dim=4):
        super().__init__()
        self.z_dim = z_dim
        self.y_dim = y_dim
        self.channel = channel
        self.fc1 = nn.Linear(self.channel * 96 * 96, 300)
        self.fc2 = nn.Linear(300 + y_dim, 300)
        self.fc3 = nn.Linear(300, 300)
        self.fc4 = nn.Linear(300, 2 * z_dim)
        self.LReLU = nn.LeakyReLU(0.2, inplace=True)
        self.net = nn.Sequential(
            nn.Linear(self.channel * 96 * 96, 900),
            nn.ELU(),
            nn.Linear(900, 300),
            nn.ELU(),
            nn.Linear(300, 2 * z_dim),
        )
        
        # self.net = nn.Sequential(
        #     nn.Linear(self.channel * 96 * 96, 900),
        #     nn.ELU(),
        #     nn.Linear(900, 300),
        # )
        
        self.fc_mu = nn.Linear(300, z_dim)
        self.fc_var = nn.Linear(300, z_dim)

    def conditional_encode(self, x, l):
        x = x.view(-1, self.channel * 96 * 96)
        x = F.elu(self.fc1(x))
        l = l.view(-1, 4)
        x = F.elu(self.fc2(torch.cat([x, l], dim=1)))
        x = F.elu(self.fc3(x))
        x = self.fc4(x)
        m, v = ut.gaussian_parameters(x, dim=1)
        return m, v

    def encode(self, x, y=None):
        xy = x if y is None else torch.cat((x, y), dim=1)
        xy = xy.view(-1, self.channel * 96 * 96)
        h = self.net(xy)
        
        # m = self.fc_mu(h)
        # v = self.fc_var(h)
        
        m, v = ut.gaussian_parameters(h, dim=1)
        
        return m, v

    
class Decoder(nn.Module):
    def __init__(self, z_dim, y_dim=0):
        super().__init__()
        self.z_dim = z_dim
        self.y_dim = y_dim
        self.net = nn.Sequential(
            nn.Linear(z_dim + y_dim, 300),
            nn.ELU(),
            nn.Linear(300, 300),
            nn.ELU(),
            nn.Linear(300, 4 * 96 * 96)
        )

    def decode(self, z, y=None):
        zy = z if y is None else torch.cat((z, y), dim=1)
        return self.net(zy)


    

    
    
class Decoder_DAG(nn.Module):
    def __init__(self, z_dim, concept, z1_dim, channel=4, y_dim=0):
        super().__init__()
        self.z_dim = z_dim
        self.z1_dim = z1_dim
        self.concept = concept
        self.y_dim = y_dim
        self.channel = channel
        # print(self.channel)
        self.elu = nn.ELU()
        self.net1 = nn.Sequential(
            nn.Linear(z1_dim + y_dim, 300),
            nn.ELU(),
            nn.Linear(300, 300),
            nn.ELU(),
            nn.Linear(300, 1024),
            nn.ELU(),
            nn.Linear(1024, self.channel * 96 * 96)
        )
        self.net2 = nn.Sequential(
            nn.Linear(z1_dim + y_dim, 300),
            nn.ELU(),
            nn.Linear(300, 300),
            nn.ELU(),
            nn.Linear(300, 1024),
            nn.ELU(),
            nn.Linear(1024, self.channel * 96 * 96)
        )
        self.net3 = nn.Sequential(
            nn.Linear(z1_dim + y_dim, 300),
            nn.ELU(),
            nn.Linear(300, 300),
            nn.ELU(),
            nn.Linear(300, 1024),
            nn.ELU(),
            nn.Linear(1024, self.channel * 96 * 96)
        )
        self.net4 = nn.Sequential(
            nn.Linear(z1_dim + y_dim, 300),
            nn.ELU(),
            nn.Linear(300, 300),
            nn.ELU(),
            nn.Linear(300, 1024),
            nn.ELU(),
            nn.Linear(1024, self.channel * 96 * 96)
        )
        self.net5 = nn.Sequential(
            nn.ELU(),
            nn.Linear(1024, self.channel * 96 * 96)
        )

        self.net6 = nn.Sequential(
            nn.Linear(z_dim, 300),
            nn.ELU(),
            nn.Linear(300, 300),
            nn.ELU(),
            nn.Linear(300, 1024),
            nn.ELU(),
            nn.Linear(1024, 1024),
            nn.ELU(),
            nn.Linear(1024, self.channel * 96 * 96)
        )

    def decode_condition(self, z, u):
        # z = z.view(-1,3*4)
        z = z.view(-1, 4 * 4)
        z1, z2, z3, z4 = torch.split(z, self.z_dim // 4, dim=1)
        # print(z1.shape)
        # exit(0)
        # print(u[:,0].reshape(1,u.size()[0]).size())
        rx1 = self.net1(
            torch.transpose(torch.cat((torch.transpose(z1, 1, 0), u[:, 0].reshape(1, u.size()[0])), dim=0), 1, 0))
        rx2 = self.net2(
            torch.transpose(torch.cat((torch.transpose(z2, 1, 0), u[:, 1].reshape(1, u.size()[0])), dim=0), 1, 0))
        rx3 = self.net3(
            torch.transpose(torch.cat((torch.transpose(z3, 1, 0), u[:, 2].reshape(1, u.size()[0])), dim=0), 1, 0))
        rx4 = self.net4(
            torch.transpose(torch.cat((torch.transpose(z4, 1, 0), u[:, 2].reshape(1, u.size()[0])), dim=0), 1, 0))
        temp = torch.cat((rx1, rx2, rx3, rx4), dim=1)
        # print(temp.shape)
        # exit(0)
        # h = self.net6(torch.cat((rx1, rx2, rx3, rx4), dim=1))

        h = (rx1 + rx2 + rx3 + rx4) / 4

        return h

    def decode_mix(self, z):
        z = z.permute(0, 2, 1)
        z = torch.sum(z, dim=2, out=None)
        # print(z.contiguous().size())
        z = z.contiguous()
        h = self.net1(z)
        return h

    def decode_union(self, z, u, y=None):

        z = z.view(-1, self.concept * self.z1_dim)
        zy = z if y is None else torch.cat((z, y), dim=1)
        if self.z1_dim == 1:
            zy = zy.reshape(zy.size()[0], zy.size()[1], 1)
            zy1, zy2, zy3, zy4 = zy[:, 0], zy[:, 1], zy[:, 2], zy[:, 3]
        else:
            zy1, zy2, zy3, zy4 = torch.split(zy, self.z_dim // self.concept, dim=1)
        rx1 = self.net1(zy1)
        rx2 = self.net2(zy2)
        rx3 = self.net3(zy3)
        rx4 = self.net4(zy4)
        h = self.net5((rx1 + rx2 + rx3 + rx4) / 4)
        return h, h, h, h, h

    def decode(self, z, u, y=None):
        z = z.view(-1, self.concept * self.z1_dim)
        h = self.net6(z)
        return h, h, h, h, h

    def decode_sep(self, z, u, y=None):
        z = z.view(-1, self.concept * self.z1_dim)
        zy = z if y is None else torch.cat((z, y), dim=1)

        if self.z1_dim == 1:
            zy = zy.reshape(zy.size()[0], zy.size()[1], 1)
            if self.concept == 4:
                zy1, zy2, zy3, zy4 = zy[:, 0], zy[:, 1], zy[:, 2], zy[:, 3]
            elif self.concept == 3:
                zy1, zy2, zy3 = zy[:, 0], zy[:, 1], zy[:, 2]
        else:
            if self.concept == 4:
                zy1, zy2, zy3, zy4 = torch.split(zy, self.z_dim // self.concept, dim=1)
            elif self.concept == 3:
                zy1, zy2, zy3 = torch.split(zy, self.z_dim // self.concept, dim=1)
        rx1 = self.net1(zy1)
        rx2 = self.net2(zy2)
        rx3 = self.net3(zy3)
        if self.concept == 4:
            rx4 = self.net4(zy4)
            h = (rx1 + rx2 + rx3 + rx4) / self.concept
        elif self.concept == 3:
            h = (rx1 + rx2 + rx3) / self.concept

        return h, h, h, h, h

    def decode_cat(self, z, u, y=None):
        z = z.view(-1, 4 * 4)
        zy = z if y is None else torch.cat((z, y), dim=1)
        zy1, zy2, zy3, zy4 = torch.split(zy, 1, dim=1)
        rx1 = self.net1(zy1)
        rx2 = self.net2(zy2)
        rx3 = self.net3(zy3)
        rx4 = self.net4(zy4)
        h = self.net5(torch.cat((rx1, rx2, rx3, rx4), dim=1))
        return h
    
    
    
class F_SCM(nn.Module):
    def __init__(self, latent_dim=4, f_dim=4):
        super().__init__()
        self.latent_dim = latent_dim
        self.f_dim = f_dim
        self.net = nn.Sequential(
            nn.Linear(self.latent_dim*self.f_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, f_dim)
        )

    def forward(self, z, z_int, mask, I=None):
        # print(z.shape)
        # exit(0)
        z_masked = torch.empty(z.size()).to(device)
        for i in range(4):
            # if 1 in mask[:, i]:
            #     eps = torch.normal(mean=torch.zeros(self.f_dim), std=torch.ones(self.f_dim)).to(device)
            #     z_masked[:, i] = self.net((z * mask[:, i]).reshape(-1, self.latent_dim*self.f_dim)) + eps
            # else:
            #     z_masked[:, i] = z[:, i]

            if I is not None:
                for j in range(z.shape[0]):
                    if I[j][0][i] == 0:
                        z_masked[j, i] = z[j, i]
                    else:
                        z_masked[j, i] = z_int[j, i]

            if 1 in mask[:, i]:
                eps = torch.normal(mean=torch.zeros(self.f_dim), std=torch.ones(self.f_dim)).to(device)
                z_masked[:, i] = self.net((z * mask[:, i]).reshape(-1, self.latent_dim*self.f_dim)) + eps
            else:
                z_masked[:, i] = z[:, i]
        # exit(0)
        # mean, std, var = torch.mean(z_masked), torch.std(z_masked), torch.var(z_masked)

        return z_masked


class MaskLayer(nn.Module):
    def __init__(self, z_dim, concept=4, z1_dim=4):
        super().__init__()
        self.z_dim = z_dim
        self.z1_dim = z1_dim
        self.concept = concept

        self.elu = nn.ELU()
        self.net1 = nn.Sequential(
            nn.Linear(z1_dim, 32),
            nn.ELU(),
            nn.Linear(32, z1_dim),
        )
        self.net2 = nn.Sequential(
            nn.Linear(z1_dim, 32),
            nn.ELU(),
            nn.Linear(32, z1_dim),
        )
        self.net3 = nn.Sequential(
            nn.Linear(z1_dim, 32),
            nn.ELU(),
            nn.Linear(32, z1_dim),
        )
        self.net4 = nn.Sequential(
            nn.Linear(z1_dim, 32),
            nn.ELU(),
            nn.Linear(32, z1_dim)
        )
        self.net = nn.Sequential(
            nn.Linear(z_dim, 32),
            nn.ELU(),
            nn.Linear(32, z_dim),
        )
        
    def masked(self, z):
        z = z.view(-1, self.z_dim)
        z = self.net(z)
        return z

    def masked_sep(self, z):
        z = z.view(-1, self.z_dim)
        z = self.net(z)
        return z


    def mix(self, z):
        zy = z.view(-1, self.concept * self.z1_dim)
        if self.z1_dim == 1:
            zy = zy.reshape(zy.size()[0], zy.size()[1], 1)
            if self.concept == 4:
                zy1, zy2, zy3, zy4 = zy[:, 0], zy[:, 1], zy[:, 2], zy[:, 3]
            elif self.concept == 3:
                zy1, zy2, zy3 = zy[:, 0], zy[:, 1], zy[:, 2]
        else:
            if self.concept == 4:
                #print(zy.shape)
                #print(len(torch.split(zy, self.z_dim // self.concept, dim=1)))
                zy1, zy2, zy3, zy4 = torch.split(zy, self.z_dim // self.concept, dim=1)
            elif self.concept == 3:
                zy1, zy2, zy3 = torch.split(zy, self.z_dim // self.concept, dim=1)
        rx1 = self.net1(zy1)
        rx2 = self.net2(zy2)
        rx3 = self.net3(zy3)
        if self.concept == 4:
            rx4 = self.net4(zy4)
            h = torch.cat((rx1, rx2, rx3, rx4), dim=1)
        elif self.concept == 3:
            h = torch.cat((rx1, rx2, rx3), dim=1)
        # print(h.size())
        return h

    
    
class MaskLayer1(nn.Module):
    def __init__(self, z_dim, concept=4, z1_dim=4):
        super().__init__()
        self.z_dim = z_dim
        self.z1_dim = z1_dim
        self.concept = concept

        self.elu = nn.ELU()
        self.net1 = nn.Sequential(
            nn.Linear(z1_dim, 32),
            nn.ELU(),
            nn.Linear(32, z1_dim),
        )
        self.net2 = nn.Sequential(
            nn.Linear(z1_dim, 32),
            nn.ELU(),
            nn.Linear(32, z1_dim),
        )
        self.net3 = nn.Sequential(
            nn.Linear(z1_dim, 32),
            nn.ELU(),
            nn.Linear(32, z1_dim),
        )
        self.net4 = nn.Sequential(
            nn.Linear(z1_dim, 32),
            nn.ELU(),
            nn.Linear(32, z1_dim)
        )
        self.net = nn.Sequential(
            nn.Linear(z_dim, 32),
            nn.ELU(),
            nn.Linear(32, z_dim),
        )
        self.net_g = nn.Sequential(
            nn.Linear(z_dim, 32),
            nn.ELU(),
            nn.Linear(32, z1_dim),
        )

    def masked(self, z):
        z = z.view(-1, self.z_dim)
        z = self.net(z)
        return z

    def masked_sep(self, z):
        z = z.view(-1, self.z_dim)
        z = self.net(z)
        return z

    def g(self, z, i=None):
        # z = z[:, :8]
        # print(z.shape)
        # exit(0)
        rx = self.net_g(z)

        # print(rx.shape)
        # exit(0)
        return rx

    def mix(self, z):
        zy = z.view(-1, self.concept * self.z1_dim)
        if self.z1_dim == 1:
            zy = zy.reshape(zy.size()[0], zy.size()[1], 1)
            if self.concept == 4:
                zy1, zy2, zy3, zy4 = zy[:, 0], zy[:, 1], zy[:, 2], zy[:, 3]
            elif self.concept == 3:
                zy1, zy2, zy3 = zy[:, 0], zy[:, 1], zy[:, 2]
        else:
            if self.concept == 4:
                #print(zy.shape)
                #print(len(torch.split(zy, self.z_dim // self.concept, dim=1)))
                zy1, zy2, zy3, zy4 = torch.split(zy, self.z_dim // self.concept, dim=1)
            elif self.concept == 3:
                zy1, zy2, zy3 = torch.split(zy, self.z_dim // self.concept, dim=1)
        rx1 = self.net1(zy1)
        rx2 = self.net2(zy2)
        rx3 = self.net3(zy3)
        if self.concept == 4:
            rx4 = self.net4(zy4)
            h = torch.cat((rx1, rx2, rx3, rx4), dim=1)
        elif self.concept == 3:
            h = torch.cat((rx1, rx2, rx3), dim=1)
        # print(h.size())
        return h
    
    

class Mix(nn.Module):
    def __init__(self, z_dim, concept, z1_dim):
        super().__init__()
        self.z_dim = z_dim
        self.z1_dim = z1_dim
        self.concept = concept

        self.elu = nn.ELU()
        self.net1 = nn.Sequential(
            nn.Linear(z1_dim, 16),
            nn.ELU(),
            nn.Linear(16, z1_dim),
        )
        self.net2 = nn.Sequential(
            nn.Linear(z1_dim, 16),
            nn.ELU(),
            nn.Linear(16, z1_dim),
        )
        self.net3 = nn.Sequential(
            nn.Linear(z1_dim, 16),
            nn.ELU(),
            nn.Linear(16, z1_dim),
        )
        self.net4 = nn.Sequential(
            nn.Linear(z1_dim, 16),
            nn.ELU(),
            nn.Linear(16, z1_dim),
        )

    def mix(self, z):
        zy = z.view(-1, self.concept * self.z1_dim)
        if self.z1_dim == 1:
            zy = zy.reshape(zy.size()[0], zy.size()[1], 1)
            zy1, zy2, zy3, zy4 = zy[:, 0], zy[:, 1], zy[:, 2], zy[:, 3]
        else:
            zy1, zy2, zy3, zy4 = torch.split(zy, self.z_dim // self.concept, dim=1)
        rx1 = self.net1(zy1)
        rx2 = self.net2(zy2)
        rx3 = self.net3(zy3)
        rx4 = self.net4(zy4)
        h = torch.cat((rx1, rx2, rx3, rx4), dim=1)
        # print(h.size())
        return h


class CausalLayer(nn.Module):
    def __init__(self, z_dim, concept=4, z1_dim=4):
        super().__init__()
        self.z_dim = z_dim
        self.z1_dim = z1_dim
        self.concept = concept

        self.elu = nn.ELU()
        self.net1 = nn.Sequential(
            nn.Linear(z1_dim, 32),
            nn.ELU(),
            nn.Linear(32, z1_dim),
        )
        self.net2 = nn.Sequential(
            nn.Linear(z1_dim, 32),
            nn.ELU(),
            nn.Linear(32, z1_dim),
        )
        self.net3 = nn.Sequential(
            nn.Linear(z1_dim, 32),
            nn.ELU(),
            nn.Linear(32, z1_dim),
        )
        self.net4 = nn.Sequential(
            nn.Linear(z1_dim, 32),
            nn.ELU(),
            nn.Linear(32, z1_dim)
        )
        self.net = nn.Sequential(
            nn.Linear(z_dim, 128),
            nn.ELU(),
            nn.Linear(128, z_dim),
        )

    def calculate(self, z, v):
        z = z.view(-1, self.z_dim)
        z = self.net(z)
        return z, v

    def masked_sep(self, z, v):
        z = z.view(-1, self.z_dim)
        z = self.net(z)
        return z, v

    def calculate_dag(self, z, v):
        zy = z.view(-1, self.concept * self.z1_dim)
        if self.z1_dim == 1:
            zy = zy.reshape(zy.size()[0], zy.size()[1], 1)
            zy1, zy2, zy3, zy4 = zy[:, 0], zy[:, 1], zy[:, 2], zy[:, 3]
        else:
            zy1, zy2, zy3, zy4 = torch.split(zy, self.z_dim // self.concept, dim=1)
        rx1 = self.net1(zy1)
        rx2 = self.net2(zy2)
        rx3 = self.net3(zy3)
        rx4 = self.net4(zy4)
        h = torch.cat((rx1, rx2, rx3, rx4), dim=1)
        # print(h.size())
        return h, v


class Attention(nn.Module):
    def __init__(self, in_features, bias=False):
        super().__init__()
        self.M = nn.Parameter(torch.nn.init.normal_(torch.zeros(in_features, in_features), mean=0, std=1))
        self.sigmd = torch.nn.Sigmoid()

    # self.M =  nn.Parameter(torch.zeros(in_features,in_features))
    # self.A = torch.zeros(in_features,in_features).to(device)

    def attention(self, z, e):
        a = z.matmul(self.M).matmul(e.permute(0, 2, 1))
        a = self.sigmd(a)
        # print(self.M)
        A = torch.softmax(a, dim=1)
        e = torch.matmul(A, e)
        return e, A


class DagLayer(nn.Linear):
    def __init__(self, in_features, out_features, A=None, i=False, bias=False):
        super(Linear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.i = i
        self.a = torch.zeros(out_features, out_features)
        self.a = self.a
        # self.a[0][1], self.a[0][2], self.a[0][3] = 1, 1, 1
        # self.a[1][2], self.a[1][3] = 1, 1
        # self.a[0, 2], self.a[1, 2], self.a[1, 3], self.a[3, 2] = 1, 1, 1, 1

        # self.a[0, 2:4] = 1
        # self.a[1, 2:4] = 1
        self.a = A

        # self.a[0, 1], self.a[1, 2], self.a[3, 2] = 1, 1, 1
        # self.a[0, 2], self.a[1, 3], self.a[2, 3] = 1, 1, 1

        # self.A = nn.Parameter(self.a)
        self.A = self.a#.to(device)

        self.b = torch.eye(out_features)
        self.b = self.b
        self.B = self.b#.to(device)
        # self.B = nn.Parameter(self.b)

        self.I = torch.eye((out_features))#.to(device)
        # self.I = nn.Parameter(torch.eye(out_features))
        # self.I.requires_grad = False
        if bias:
            self.bias = Parameter(torch.Tensor(out_features))
        else:
            self.register_parameter('bias', None)

    def mask_z(self, x, i):
        self.B = self.A.to(x.device)

        # x = torch.mul(self.B[:, i].reshape(4, 1).clone(), x.clone())
        x = torch.mul((self.B + self.I.to(x.device))[:, i].reshape(4, 1).clone(), x.clone())

        # x = torch.matmul(self.B.t().clone(), x.clone())
        # print(x.shape)
        # x = torch.matmul(self.B.t(), x)

        return x
    
    def mask_z_orig(self,x):
        self.B = self.A.to(x.device)
        #if self.i:
        #    x = x.view(-1, x.size()[1], 1)
        #    x = torch.matmul((self.B+0.5).t().int().float(), x)
        #    return x
        x = torch.matmul(self.B.t().float(), x)
        return x

    def mask_z_learn(self, x, i):
        self.B = self.A.to(x.device)

        # x = F.linear(x.clone(), (self.B + self.I)[:, i].reshape(4, 1).clone(), self.bias)
        x = torch.mul((self.B + self.I.to(x.device))[:, i].reshape(4, 1).clone(), x.clone())

        return x


    def mask_u(self, x):
        self.B = self.A.to(x.device)
        # if self.i:
        #    x = x.view(-1, x.size()[1], 1)
        #    x = torch.matmul((self.B+0.5).t().int().float(), x)
        #    return x
        x = x.view(-1, x.size()[1], 1)
        x = torch.matmul(self.B.t(), x)
        return x

    def inv_cal(self, x, v):
        if x.dim() > 2:
            x = x.permute(0, 2, 1)
        x = F.linear(x, self.I - self.A, self.bias)

        if x.dim() > 2:
            x = x.permute(0, 2, 1).contiguous()
        return x, v

    def calculate_dag(self, x, v):
        # print(self.A)
        # x = F.linear(x, torch.inverse((torch.abs(self.A))+self.I), self.bias)
        self.A = self.A.to(x.device)
        if x.dim() > 2:
            x = x.permute(0, 2, 1)
        x = F.linear(x, torch.inverse(self.I - self.A.t()), self.bias)
        # print(x.size())

        if x.dim() > 2:
            x = x.permute(0, 2, 1).contiguous()
        return x, v

    def calculate_cov(self, x, v):
        # print(self.A)
        v = ut.vector_expand(v)
        # x = F.linear(x, torch.inverse((torch.abs(self.A))+self.I), self.bias)
        x = dag_left_linear(x, torch.inverse(self.I - self.A), self.bias)
        v = dag_left_linear(v, torch.inverse(self.I - self.A), self.bias)
        v = dag_right_linear(v, torch.inverse(self.I - self.A), self.bias)
        # print(v)
        return x, v

    def calculate_gaussian_ini(self, x, v):
        print(self.A)
        # x = F.linear(x, torch.inverse((torch.abs(self.A))+self.I), self.bias)

        if x.dim() > 2:
            x = x.permute(0, 2, 1)
            v = v.permute(0, 2, 1)
        x = F.linear(x, torch.inverse(self.I - self.A), self.bias)
        v = F.linear(v, torch.mul(torch.inverse(self.I - self.A), torch.inverse(self.I - self.A)), self.bias)
        if x.dim() > 2:
            x = x.permute(0, 2, 1).contiguous()
            v = v.permute(0, 2, 1).contiguous()
        return x, v

    # def encode_
    def forward(self, x):
        # x = x * torch.inverse((self.A) + self.I)

        if x.dim() > 2:
            x = x.permute(0, 2, 1)

        x = torch.matmul(x, torch.inverse(self.I.to(x.device) - self.A.t().to(x.device)).t())

        if x.dim() > 2:
            x = x.permute(0, 2, 1).contiguous()

        return x

    def calculate_gaussian(self, x, v):
        print(self.A)
        # x = F.linear(x, torch.inverse((torch.abs(self.A))+self.I), self.bias)

        if x.dim() > 2:
            x = x.permute(0, 2, 1)
            v = v.permute(0, 2, 1)
        x = dag_left_linear(x, torch.inverse(self.I - self.A), self.bias)
        v = dag_left_linear(v, torch.inverse(self.I - self.A), self.bias)
        v = dag_right_linear(v, torch.inverse(self.I - self.A), self.bias)
        if x.dim() > 2:
            x = x.permute(0, 2, 1).contiguous()
            v = v.permute(0, 2, 1).contiguous()
        return x, v

    

class DagLayerOrig(nn.Linear):
	def __init__(self, in_features, out_features,i = False, bias=False):
		super(Linear, self).__init__()
		self.in_features = in_features
		self.out_features = out_features
		self.i = i
		# self.a = 0.5*torch.ones(out_features,out_features)
		self.a = torch.zeros(out_features,out_features)
		self.a = self.a
		#self.a[0][1], self.a[0][2], self.a[0][3] = 1,1,1
		#self.a[1][2], self.a[1][3] = 1,1

		self.a[0, 2:4], self.a[1, 2:4] = 1, 1

		# self.a[0, 2], self.a[1, 2], self.a[1, 3], self.a[3, 2] = 1, 1, 1, 1

		# self.a[0, 1], self.a[1, 2], self.a[3, 2] = 1, 1, 1
		# self.a[0, 2], self.a[2, 3], self.a[1, 3] = 1, 1, 1

		# self.A = nn.Parameter(self.a)
		self.A = self.a#.to(device)

		self.b = torch.eye(out_features)
		self.b = self.b
		# self.B = nn.Parameter(self.b)
		self.B = self.b#.to(device)

		self.I = torch.eye(out_features)#.to(device)
		# self.I = nn.Parameter(torch.eye(out_features))
		# self.I.requires_grad=False
		if bias:
			self.bias = Parameter(torch.Tensor(out_features))
		else:
			self.register_parameter('bias', None)

	def mask_z(self,x):
		self.B = self.A.to(x.device)
		#if self.i:
		#    x = x.view(-1, x.size()[1], 1)
		#    x = torch.matmul((self.B+0.5).t().int().float(), x)
		#    return x
		x = torch.matmul(self.B.t(), x)
		return x

	def mask_u(self,x):
		self.B = self.A.to(x.device)
		#if self.i:
		#    x = x.view(-1, x.size()[1], 1)
		#    x = torch.matmul((self.B+0.5).t().int().float(), x)
		#    return x
		x = x.view(-1, x.size()[1], 1)
		x = torch.matmul(self.B.t(), x)
		return x

	def inv_cal(self, x,v):
		if x.dim()>2:
			x = x.permute(0,2,1)
		x = F.linear(x, self.I - self.A, self.bias)

		if x.dim()>2:
			x = x.permute(0,2,1).contiguous()
		return x,v

	def calculate_dag(self, x):
		#print(self.A)
		#x = F.linear(x, torch.inverse((torch.abs(self.A))+self.I), self.bias)

		if x.dim()>2:
			x = x.permute(0,2,1)
		x = F.linear(x, torch.inverse(self.I.to(x.device) - self.A.t().to(x.device)), self.bias)
		#print(x.size())

		if x.dim()>2:
			x = x.permute(0,2,1).contiguous()
		return x

	def calculate_cov(self, x, v):
		#print(self.A)
		v = ut.vector_expand(v)
		#x = F.linear(x, torch.inverse((torch.abs(self.A))+self.I), self.bias)
		x = dag_left_linear(x, torch.inverse(self.I - self.A), self.bias)
		v = dag_left_linear(v, torch.inverse(self.I - self.A), self.bias)
		v = dag_right_linear(v, torch.inverse(self.I - self.A), self.bias)
		#print(v)
		return x, v

	def calculate_gaussian_ini(self, x, v):
		print(self.A)
		#x = F.linear(x, torch.inverse((torch.abs(self.A))+self.I), self.bias)

		if x.dim()>2:
			x = x.permute(0,2,1)
			v = v.permute(0,2,1)
		x = F.linear(x, torch.inverse(self.I - self.A), self.bias)
		v = F.linear(v, torch.mul(torch.inverse(self.I - self.A),torch.inverse(self.I - self.A)), self.bias)
		if x.dim()>2:
			x = x.permute(0,2,1).contiguous()
			v = v.permute(0,2,1).contiguous()
		return x, v
	#def encode_
	def forward(self, x):
		# x = x * torch.inverse((self.A)+self.I)

		if x.dim() > 2:
			x = x.permute(0, 2, 1)

		x = torch.matmul(x, torch.inverse(self.I.to(x.device) - self.A.t().to(x.device)).t())

		if x.dim() > 2:
			x = x.permute(0, 2, 1).contiguous()

		return x
	def calculate_gaussian(self, x, v):
		print(self.A)
		#x = F.linear(x, torch.inverse((torch.abs(self.A))+self.I), self.bias)

		if x.dim()>2:
			x = x.permute(0,2,1)
			v = v.permute(0,2,1)
		x = dag_left_linear(x, torch.inverse(self.I - self.A), self.bias)
		v = dag_left_linear(v, torch.inverse(self.I - self.A), self.bias)
		v = dag_right_linear(v, torch.inverse(self.I - self.A), self.bias)
		if x.dim()>2:
			x = x.permute(0,2,1).contiguous()
			v = v.permute(0,2,1).contiguous()
		return x, v
    
    
    
    

    
# [(in - k + 2p)/s] + 1
class ConvEncoder(nn.Module):
    def __init__(self, out_dim=None):
        super().__init__()
        # init 128*128
        # 64x64x32, 32x32x64, 16x16x64, 8x8x64, 4x4x256, 4x4x3 (want
        # init 96*96
        self.conv1 = torch.nn.Conv2d(3, 32, 4, 2, 1)  # 48*48
        self.conv2 = torch.nn.Conv2d(32, 64, 4, 2, 1, bias=False)  # 24*24
        self.conv3 = torch.nn.Conv2d(64, 1, 4, 2, 1, bias=False)
        # self.conv4 = torch.nn.Conv2d(128, 1, 1, 1, 0) # 54*44

        self.LReLU = torch.nn.LeakyReLU(0.2, inplace=True)
        self.convm = torch.nn.Conv2d(1, 1, 4, 2, 1)
        self.convv = torch.nn.Conv2d(1, 1, 4, 2, 1)
        self.mean_layer = nn.Sequential(
            torch.nn.Linear(8 * 8, out_dim)
        )  # 12*12
        self.var_layer = nn.Sequential(
            torch.nn.Linear(8 * 8, out_dim)
        )
        # self.fc1 = torch.nn.Linear(6*6*128, 512)
        self.conv6 = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),
            nn.ReLU(True),
            nn.Conv2d(32, 64, 4, 2, 1),
            nn.ReLU(True),
            nn.Conv2d(64, 64, 4, 2, 1),
            nn.ReLU(True),
            nn.Conv2d(64, 64, 4, 2, 1),
            nn.ReLU(True),
            nn.Conv2d(64, 256, 4, 2, 1), # 4x4x256
            nn.ReLU(True),
            nn.Conv2d(256, 64, 4, 2, 1) # 2x2x64
        )

    def encode(self, x):
        x = self.LReLU(self.conv1(x))
        x = self.LReLU(self.conv2(x))
        x = self.LReLU(self.conv3(x))
        # x = self.LReLU(self.conv4(x))
        # print(x.size())
        hm = self.convm(x)
        # print(hm.size())
        hm = hm.view(-1, 8 * 8)
        hv = self.convv(x)
        hv = hv.view(-1, 8 * 8)
        mu, var = self.mean_layer(hm), self.var_layer(hv)
        var = F.softplus(var) + 1e-8
        # var = torch.reshape(var, [-1, 16, 16])
        # print(mu.size())
        return mu, var

    def encode_simple(self, x):
        x = self.conv6(x)
        x = x.reshape(x.shape[0], 256)
        # print(x.shape)
        # exit(0)
        m, v = ut.gaussian_parameters(x, dim=1)
        # print(m.size())
        return m, v

# [(in - k + 2p)/s] + 1 = out
class ConvDecoder(nn.Module):
    def __init__(self, z2_dim):
        super().__init__()
        self.z2_dim = z2_dim
        
        self.net6 = nn.Sequential(
            nn.Conv2d(self.z2_dim, 128, 1), # 1-1+0 / 1 = 1x1x128
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64, 4), # (128 - 1)
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 32, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 32, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 3, 4, 2, 1)
        )

    def decode_sep(self, x):
        return None

    def decode(self, z):
        z = z.view(-1, self.z2_dim, 1, 1)
        z = self.net6(z)
        return z


class ConvDec(nn.Module):
    def __init__(self, concept, z1_dim, z_dim):
        super().__init__()
        self.concept = concept
        self.z1_dim = z1_dim
        self.z_dim = z_dim
        self.net1 = ConvDecoder(z1_dim)
        self.net2 = ConvDecoder(z1_dim)
        self.net3 = ConvDecoder(z1_dim)
        self.net4 = ConvDecoder(z1_dim)
        self.net5 = nn.Sequential(
            nn.Linear(16, 512),
            nn.BatchNorm1d(512),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024)
        )
        self.net6 = nn.Sequential(
            nn.Conv2d(16, 128, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64, 4),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 32, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 32, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 3, 4, 2, 1)
        )

    def decode_sep(self, z, u=None, y=None):
        z = z.view(-1, self.concept * self.z1_dim)

        zy = z if y is None else torch.cat((z, y), dim=1)
        zy1, zy2, zy3, zy4 = torch.split(zy, self.z_dim // self.concept, dim=1)
        rx1 = self.net1.decode(zy1)
        # print(rx1.size())
        rx2 = self.net2.decode(zy2)
        rx3 = self.net3.decode(zy3)
        rx4 = self.net4.decode(zy4)
        z = (rx1 + rx2 + rx3 + rx4) / 4
        return z, z, z, z, z

    def decode(self, z, u=None, y=None):
        z = z.view(-1, self.concept * self.z1_dim, 1, 1)
        z = self.net6(z)
        # print(z.size())

        return z










#############################################################################################################
############################################# CELEBA ENC/DEC ################################################
#############################################################################################################

class CelebAConvEncoder(nn.Module):
    def __init__(self, latent_dim, in_channels=3, out_dim=None):
        super().__init__()
        self.latent_dim = latent_dim
        # 128x128
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, 4, 2, 1),            # 64x64x32
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, 4, 2, 1),                     # 32x32x64
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 64, 4, 2, 1),                     # 16x16x64
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1),                    # 8x8x128
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 128, 4, 2, 1),                   # 4x4x128
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 1),                      # 1x1x256
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 1)                    # 1x1x512
        )




        modules = []

        hidden_dims = [32, 64, 128, 256, 512, 512, 512]

        # in: 128x128x3, out: 64x64x32
        # in: 64x64x32, out: 32x32x64
        # out: 16x16x128
        # out: 8x8x256
        # out: 4x4x512

        # Build Encoder
        for h_dim in hidden_dims:
            modules.append(
                nn.Sequential(
                    nn.Conv2d(in_channels, out_channels=h_dim,
                              kernel_size=4, stride=2, padding=1),
                    nn.BatchNorm2d(h_dim),
                    nn.LeakyReLU(0.2, inplace=True))
            )
            in_channels = h_dim

        self.encoder = nn.Sequential(*modules)
        self.fc_mu = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_var = nn.Linear(hidden_dims[-1], latent_dim)


    def encode(self, x):
        z = self.conv(x)
        z = z.view(-1, 512)

        # Split the result into mu and var components
        # of the latent Gaussian distribution
        mu = self.fc_mu(z)
        var = self.fc_var(z)
        var = F.softplus(var) + 1e-8

        return mu, var


class CelebAConvDecoder(nn.Module):
    def __init__(self, latent_dim, out_channels=3, out_dim=None):
        super().__init__()
        self.latent_dim = latent_dim


        self.convT = nn.Sequential(
            nn.Conv2d(latent_dim, 512, 1),                      # 1x1x512
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(512, 256, 4),                    # 4x4x256
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),              # 8x8x128
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(128, 128, 4, 2, 1),              # 16x16x128
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),               # 32x32x64
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(64, 64, 4, 2, 1),                # 64x64x64
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),                # 128x128x32
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(32, out_channels, 1),         # 128x128x3
        )


        modules = []

        hidden_dims = [32, 64, 128, 256, 512, 512, 512]

        # in: 128x128x3, out: 64x64x32
        # in: 64x64x32, out: 32x32x64
        # out: 16x16x128
        # out: 8x8x256
        # out: 4x4x512

        self.decoder_input = nn.Linear(latent_dim, hidden_dims[-1])

        hidden_dims.reverse()

        # in: 4x4x512, out: 8x8x256
        # in: 8x8x256, out: 16x16x128
        for i in range(len(hidden_dims) - 1):
            modules.append(
                nn.Sequential(
                    nn.ConvTranspose2d(hidden_dims[i],
                                       hidden_dims[i + 1],
                                       kernel_size=4,
                                       stride=2,
                                       padding=1),
                    nn.BatchNorm2d(hidden_dims[i + 1]),
                    nn.LeakyReLU(0.2, inplace=True))
            )

        self.decoder = nn.Sequential(*modules)

        self.final_layer = nn.Sequential(
            nn.ConvTranspose2d(hidden_dims[-1],
                               hidden_dims[-1],
                               kernel_size=4,
                               stride=2,
                               padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_dims[-1]),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(hidden_dims[-1], out_channels=out_channels,
                      kernel_size=4, padding=1),
            nn.Tanh())

    def decode(self, z):
        z = z.view(-1, self.latent_dim, 1, 1)
        z = self.convT(z)
        return z

    # def decode(self, z):
    #     z = self.decoder_input(z)
    #     z = z.view(-1, 512, 1, 1)
    #     x = self.decoder(z)
    #     x = self.final_layer(x)
    #
    #     return x


class CelebAConvDec(nn.Module):
    def __init__(self, latent_dim, out_dim=None):
        super().__init__()
        self.concept = 4
        self.z_dim = latent_dim
        self.z1_dim = self.z_dim // self.concept

        self.net1 = CelebAConvDecoder(self.z1_dim)
        self.net2 = CelebAConvDecoder(self.z1_dim)
        self.net3 = CelebAConvDecoder(self.z1_dim)
        self.net4 = CelebAConvDecoder(self.z1_dim)

    def decode_sep(self, z, u=None, y=None):
        z = z.view(-1, self.concept * self.z1_dim)  # 16x64

        zy = z if y is None else torch.cat((z, y), dim=1)
        # print(zy.shape)
        zy1, zy2, zy3, zy4 = torch.split(zy, self.z_dim // self.concept, dim=1)  # each is 16x16
        rx1 = self.net1.decode(zy1)
        # print(f"Hi: {rx1.size()}")
        rx2 = self.net2.decode(zy2)
        rx3 = self.net3.decode(zy3)
        rx4 = self.net4.decode(zy4)
        # z = torch.cat((rx1, rx2, rx3, rx4), dim=0)
        z = (rx1+rx2+rx3+rx4)/4
        # print(z.shape)
        # sys.exit(0)
        return z, z, z, z, z






class ConvEncoderPend(nn.Module):
    def __init__(self, latent_dim, in_channel=3, out_dim=None):
        super().__init__()
        # init 96*96
        self.conv1 = torch.nn.Conv2d(in_channel, 24, 4, 2, 1)  # 48*48
        self.conv2 = torch.nn.Conv2d(24, 48, 4, 2, 1, bias=False)  # 24*24
        self.conv3 = torch.nn.Conv2d(48, 1, 4, 2, 1, bias=False)  # 12x12
        self.conv4 = torch.nn.Conv2d(1, 1, 3, 1, bias=False)
        # self.conv4 = torch.nn.Conv2d(128, 1, 1, 1, 0) # 54*44

        self.LReLU = torch.nn.LeakyReLU(0.2, inplace=True)
        self.convm = torch.nn.Conv2d(1, 1, 3, 1)  # 6x6 - BUT, changed padding from 1 to 3 in order to make this 8x8
        self.convv = torch.nn.Conv2d(1, 1, 3, 1)  # 6x6
        self.mean_layer = nn.Sequential(
            torch.nn.Linear(8*8, latent_dim)
        )  # 12*12
        self.var_layer = nn.Sequential(
            torch.nn.Linear(8*8, latent_dim)
        )

        # self.conv = nn.Sequential(
        #     nn.Conv2d(3, 32, 4, 2, 1),  # 48x48
        #     nn.ReLU(True),
        #     nn.Conv2d(32, 32, 4, 2, 1),  # 24x24
        #     nn.ReLU(True),
        #     nn.Conv2d(32, 64, 4, 2, 1),  # 12x12
        #     nn.ReLU(True),
        #     nn.Conv2d(64, 64, 4, 2, 1),  # 6x6
        #     nn.ReLU(True),
        #     nn.Conv2d(64, 64, 4, 2, 1),  # 3x3
        #     nn.ReLU(True),
        #     nn.Conv2d(64, 256, 4, 1),  # 2x2
        #     nn.ReLU(True),
        #     nn.Conv2d(256, 128, 1)  # 2x2
        # )


        self.conv = nn.Sequential(
            nn.Conv2d(in_channel, 24, 4, 2, 1),  # 48x48x24
            nn.LeakyReLU(0.2),
            nn.Conv2d(24, 24, 4, 2, 1),  # 24x24x24
            nn.LeakyReLU(0.2),
            nn.Conv2d(24, 48, 4, 2, 1),  # 12x12x48
            nn.LeakyReLU(0.2),
            nn.Conv2d(48, 48, 4, 2, 1),  # 6x6x48
            nn.LeakyReLU(0.2),
            nn.Conv2d(48, 48, 4, 2, 1),  # 3x3x48
            nn.LeakyReLU(0.2),
            nn.Conv2d(48, 96, 3, 1),  # 3x3x48
            nn.LeakyReLU(0.2),
            nn.Conv2d(96, latent_dim*2, 4, 2, 2) # 1x1x32
        )

    def encode(self, x):
        # print(x.shape)
        # sys.exit(0)
        x = self.LReLU(self.conv1(x))
        x = self.LReLU(self.conv2(x))
        x = self.LReLU(self.conv3(x))
        x = self.LReLU(self.conv4(x))

        # x = self.LReLU(self.conv4(x))
        # print(x.size())
        hm = self.convm(x)
        # print(hm.size())
        hm = hm.view(-1, 8 * 8)
        # print(hm.size())
        hv = self.convv(x)
        hv = hv.view(-1, 8 * 8)

        # print(hm.shape)
        # sys.exit(0)
        mu, var = self.mean_layer(hm), self.var_layer(hv)
        var = F.softplus(var) + 1e-8
        # var = torch.reshape(var, [-1, 16, 16])
        # print(mu.size())
        return mu, var

    def encode_simple(self, x):
        x = self.conv(x)
        # print(x.shape)
        # sys.exit(0)
        # x = x.view(-1, 96)
        # x = self.fc(x)
        # print(x.shape)
        m, v = ut.gaussian_parameters(x, dim=1)

        return m, v




# CONVOLUTIONAL DECODER LAYER
class ConvDecoderPend(nn.Module):
    def __init__(self, latent_dim, channels=3, out_dim=None):
        super().__init__()

        self.net6 = nn.Sequential(
            nn.Conv2d(latent_dim, 96, 1),  # 1x1x96
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(96, 48, 4),  # 3x3x48
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(48, 48, 2, 2, 1),  # 6x6
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(48, 24, 4, 2, 1),  # 12x12
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(24, 24, 4, 2, 1),  # 24x24x24
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(24, 24, 4, 2, 1),  # 48x48x12
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(24, channels, 4, 2, 1),  # 96x96x4
        )

        # self.net6 = nn.Sequential(
        #     nn.Conv2d(latent_dim, 96, 1), # 1x1x96
        #     nn.LeakyReLU(0.2),
        #     nn.ConvTranspose2d(96, 48, 3, 1), # 3x3x48
        #     nn.LeakyReLU(0.2),
        #     nn.ConvTranspose2d(48, 48, 3, 2, 1), # 6x6
        #     nn.LeakyReLU(0.2),
        #     nn.ConvTranspose2d(48, 24, 3, 2, 1), # 12x12
        #     nn.LeakyReLU(0.2),
        #     nn.ConvTranspose2d(24, 24, 3, 2, 1), # 24x24x24
        #     nn.LeakyReLU(0.2),
        #     nn.ConvTranspose2d(24, 24, 3, 2, 1), # 48x48x12
        #     nn.LeakyReLU(0.2),
        #     nn.ConvTranspose2d(24, channels, 3, 2, 1), # 96x96x4
        # )

    def decode_sep(self, x):
        return None

    def decode(self, z):
        z = z.view(-1, z.shape[1], 1, 1)
        z = self.net6(z)
        return z



class ConvDecPend(nn.Module):
    def __init__(self, latent_dim, channels=3, out_dim=None):
        super().__init__()
        self.concept = 4
        self.z_dim = latent_dim
        self.z1_dim = self.z_dim // self.concept

        self.net1 = ConvDecoderPend(self.z1_dim, channels)
        self.net2 = ConvDecoderPend(self.z1_dim, channels)
        self.net3 = ConvDecoderPend(self.z1_dim, channels)
        self.net4 = ConvDecoderPend(self.z1_dim, channels)
        self.net5 = nn.Sequential(
            nn.Linear(16, 512),
            nn.BatchNorm1d(512),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024)
        )
        self.net6 = nn.Sequential(
            nn.Conv2d(16, 128, 1),  # 4x4
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64, 4),  # 1x1
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 32, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 32, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 3, 4, 2, 1)
        )

    def decode_sep(self, z, u=None, y=None):
        z = z.view(-1, self.concept * self.z1_dim)  # 16x64

        zy = z if y is None else torch.cat((z, y), dim=1)
        # print(zy.shape)
        zy1, zy2, zy3, zy4 = torch.split(zy, self.z_dim // self.concept, dim=1)  # each is 16x16
        # print(zy1.shape)
        # sys.exit(0)
        rx1 = self.net1.decode(zy1)
        # print(rx1.shape)
        # sys.exit(0)
        # print(f"Hi: {rx1.size()}")
        rx2 = self.net2.decode(zy2)
        rx3 = self.net3.decode(zy3)
        rx4 = self.net4.decode(zy4)
        # z = torch.cat((rx1, rx2, rx3, rx4), dim=0)
        z = (rx1+rx2+rx3+rx4)/4
        # print(z.shape)
        # sys.exit(0)
        return z, z, z, z, z

    def decode(self, z, u=None, y=None):
        z = z.view(-1, self.concept * self.z1_dim, 1, 1)
        z = self.net6(z)
        print(z.size())

        return z


In [ ]:
import sys
sys.path.append('../')
import os
import torch
import numpy as np
from codebase import utils as ut
from utils import get_batch_unin_dataset_withlabel
from torchvision.utils import save_image
from models.icm_vae import ICM_VAE

MAX_EPOCHS=101
DATA="flow"
SAVE_DIR="icm_vae_recon"
NAME="icm_vae_cdp"
DATASET_DIR='../data/flow_noise'
RUN=0
TRAIN=1
ITER_SAVE=5
device = torch.device("cuda:3" if(torch.cuda.is_available()) else "cpu")

torch.manual_seed(42)

layout = [
	('model={:s}',  str(NAME)),
	('run={:04d}', RUN),
	('toy={:s}', str(DATA) + '_' + str(NAME))
]
model_name = '_'.join([t.format(v) for (t, v) in layout])
print('Model name:', model_name)

if not os.path.exists(f'./results/{DATA}/{DATA}_{NAME}_reconstructions/'):
	os.makedirs(f'./results/{DATA}/{DATA}_{NAME}_reconstructions/')


def save_model_by_name(model, global_step):
	save_dir = os.path.join('checkpoints', model.name)
	if not os.path.exists(save_dir):
		os.makedirs(save_dir)
	file_path = os.path.join(save_dir, 'model-{:05d}.pt'.format(global_step))
	state = model.state_dict()
	torch.save(state, file_path)
	print('Saved to {}'.format(file_path))


C = torch.tensor([[0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 0, 1], [0, 0, 0, 0]])
scale = np.array([[20,15],[10.5, 4.5],[2,2],[59.5,26.5]])

icm_vae = ICM_VAE(name=NAME + '_' + DATA, z_dim=16, z1_dim=4, z2_dim=4, C=C, scale=scale).to(device)
dataset_dir = '../../data/orig_data/flow_noise'
train_dataset = get_batch_unin_dataset_withlabel(DATASET_DIR, 64)
optimizer = torch.optim.Adam(icm_vae.parameters(), lr=1e-3, betas=(0.9, 0.999))


def linear_scheduler(step, total_steps, initial, final):
    """Linear scheduler"""

    if step >= total_steps:
        return final
    if step <= 0:
        return initial
    if total_steps <= 1:
        return final

    t = step / (total_steps - 1)
    return (1.0 - t) * initial + t * final



for epoch in range(MAX_EPOCHS):
	icm_vae.train()
	total_loss = 0
	total_rec = 0
	total_kl = 0
	for X, l in train_dataset:
		optimizer.zero_grad()
		#u = torch.bernoulli(u.to(device).reshape(u.size(0), -1))
		X = X.to(device)
		L, kl, rec, reconstructed_image, z, cp_m = icm_vae.forward(X,l,sample = False)
   
		L.backward()
		optimizer.step()
		#optimizer.zero_grad()

		total_loss += L.item()
		total_kl += kl.item() 
		total_rec += rec.item() 

		m = len(train_dataset)
		save_image(X[0], f'./results/{DATA}/{DATA}_{NAME}_reconstructions/true_{epoch}.png')
		save_image(reconstructed_image[0], f'./results/{DATA}/{DATA}_{NAME}_reconstructions/reconstructed_{epoch}.png')

	beta = linear_scheduler(epoch, 94, 0.0, 1.2)
	icm_vae.beta = beta
    
	alpha = linear_scheduler(epoch, 94, 0.0, 0.1)
	icm_vae.alpha = alpha

	if epoch % 1 == 0:
		print(str(epoch)+' loss:'+str(total_loss/m)+' kl:'+str(total_kl/m)+' rec:'+str(total_rec/m)+'m:' + str(m))

	if epoch % ITER_SAVE == 0:
		ut.save_model_by_name(icm_vae, epoch)















In [ ]:
import sys
sys.path.append('../')
import os
import torch
import numpy as np
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
from codebase import utils as ut
from codebase import metrics as mt
from utils import get_batch_unin_dataset_withlabel
from torchvision.utils import save_image
from models.icm_vae import ICM_VAE


DATA="flow"
SAVE_DIR="icm_vae_recon"
NAME="icm_vae_cdp"
DATASET_DIR='../data/flow_noise'
RUN=0
TRAIN=1
ITER_SAVE=5
device = torch.device("cuda:2" if(torch.cuda.is_available()) else "cpu")

layout = [
	('model={:s}',  str(NAME)),
	('run={:04d}', RUN),
	('toy={:s}', str(DATA) + '_' + str(NAME))
]
model_name = '_'.join([t.format(v) for (t, v) in layout])
print('Model name:', model_name)


if not os.path.exists(f'./results/{DATA}/{DATA}_{NAME}_inference/'):
	os.makedirs(f'./results/{DATA}/{DATA}_{NAME}_inference/')


def save_model_by_name(model, global_step):
	save_dir = os.path.join('checkpoints', model.name)
	if not os.path.exists(save_dir):
		os.makedirs(save_dir)
	file_path = os.path.join(save_dir, 'model-{:05d}.pt'.format(global_step))
	state = model.state_dict()
	torch.save(state, file_path)
	print('Saved to {}'.format(file_path))


C = torch.tensor([[0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 0, 1], [0, 0, 0, 0]])
scale = np.array([[20,15],[10.5, 4.5],[2,2],[59.5,26.5]])
icm_vae = ICM_VAE(name=NAME + '_' + DATA, z_dim=16, z1_dim=4, z2_dim=4, C=C, scale=scale).to(device)
ut.load_model_by_name(icm_vae, 100)

train_dataset = get_batch_unin_dataset_withlabel(DATASET_DIR, 64, dataset="train")
test_dataset = get_batch_unin_dataset_withlabel(DATASET_DIR, 64, dataset="test")

icm_vae.eval()
rep_train = np.empty((5254, 16))
y_train = np.empty((5254, 4))
for batch_idx, (X, u) in enumerate(train_dataset):
    #u = torch.bernoulli(u.to(device).reshape(u.size(0), -1))
    X = X.to(device)
    u = u.to(device)
    L, kl, rec, reconstructed_image, z, cp_m = icm_vae.forward(X,u,sample = False)
    z = z.reshape(-1, 16)
    rep_train[batch_idx*64:(batch_idx*64)+z.shape[0], :] = z.cpu().detach().numpy()
    y_train[batch_idx*64:(batch_idx*64)+u.shape[0], :] = u.cpu().detach().numpy()


icm_vae.eval()
total_loss = 0
total_rec = 0
total_kl = 0
rep_test = np.empty((1620, 16))
y_test = np.empty((1620, 4))
for batch_idx, (X, u) in enumerate(test_dataset):
    X = X.to(device)
    u = u.to(device)
    L, kl, rec, reconstructed_image, z, cp_m = icm_vae.forward(X,u,sample = False)
    z = z.reshape(-1, 16)
    rep_test[batch_idx*64:(batch_idx*64)+z.shape[0], :] = z.cpu().detach().numpy()
    y_test[batch_idx*64:(batch_idx*64)+u.shape[0], :] = u.cpu().detach().numpy()

    m = len(test_dataset)
    save_image(X, f'./results/{DATA}/{DATA}_{NAME}_inference/true.png')
    save_image(reconstructed_image, f'./results/{DATA}/{DATA}_{NAME}_inference/reconstructed.png')



scores, importance_matrix, code_importance = mt._compute_dci(rep_train.T, y_train.T, rep_test.T, y_test.T)
irs_score = mt.compute_irs(rep_train.T, y_train.T)

print(f'DCI Scores: {scores}')
print(f'Importances: {importance_matrix}')
print(f'IRS Score: {irs_score}')






























